## Installs

In [ ]:
# pip install python-dotenv
# pip install business_landscape_pycelonis --extra-index-url https://pypi.celonis.cloud/c591f5a8f4874916a268b4911580bebb --extra-index-url https://pypi.celonis.cloud/

# Inputs

### With .env file

These can be input directly here in case the env file method is not working. Note: In the MLWB dealing with hidden files can be inconvenient, so you can just name the file env instead of .env and indicate the name inside the load_doatenv function here.

In [ ]:
import os
from dotenv import load_dotenv

if not load_dotenv('.env'):
    load_dotenv('env')

url=os.getenv('url') or ""

# App key (assumes USER_KEY)
apiKey=os.getenv('apiKey') or ""

# ID for data pool you are pulling data model from
sourceDataPoolID=os.getenv('sourceDataPoolID') or "" 

# Case Centric data model you want to convert to OCPM
dataModelID=os.getenv('dataModelID') or ""

# ID for data pool you want to migrate into (where you want to create the OCDM) 
targetDataPoolID=os.getenv('targetDataPoolID') or ""

# Set Up for Logger
team_name =  os.getenv('team_name') or ""
process_name = os.getenv('process_name') or ""

In [ ]:
url

# Function Definitions

### Logger

In [ ]:
import logging
from datetime import datetime

In [ ]:
def setup_logger():
    
    # Create a 'logs' folder if it doesn't exist
    if not os.path.exists('logs'):
        os.makedirs('logs')

    # Generate a timestamp for the log file name
    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    
    # Define the log file name based on timestamp and workspace name
    log_filename = f'logs/'
    if team_name:
        log_filename += f'{team_name}_'
    if process_name:
        log_filename += f'{process_name}_'
    
    log_filename += f'{timestamp}.log'
    
     # Create a custom logger
    logger = logging.getLogger(f'{team_name}_{process_name}_{timestamp}')  # Unique ID for this logger
    logger.setLevel(logging.INFO)  # Set logging level to INFO
    
    # Create a file handler
    handler = logging.FileHandler(log_filename)
    handler.setLevel(logging.INFO)
    
    # Create a logging format
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
    handler.setFormatter(formatter)
    
    # Add the handler to the logger
    logger.addHandler(handler)
    
    # Prevent propagation of this logger to the root logger to avoid capturing external logs
    logger.propagate = False
    
    # Optionally return the logger and log file name
    return logger

def log_message(*messages):
    joint_message = " ".join(messages)
    log.info(joint_message)  # Log the message with INFO level
    print(joint_message)  # Optionally still print to console if needed

### Migration Functions

In [ ]:
from pycelonis import get_celonis
from business_landscape_pycelonis.service import BusinessLandscapePycelonis, Pagination
from datetime import datetime
import re 
import json
import pandas as pd

In [ ]:
def fixJSON(response_body):
    if type(response_body) == dict:
        keys = list(response_body.keys())
        for key in keys:
            key_split = key.split('_')
            res = key_split[0] + ''.join(map(lambda x: x.title(), key_split[1:]))
            request_part = response_body[key]
            if request_part == None:
                request_part = ''
            if request_part == 'false':
                request_part = False
            if request_part == 'true':
                request_part = True
            new_request_part = fixJSON(request_part)
            response_body[res] = new_request_part
            if key != res:
                del response_body[key]
    elif type(response_body) == list:
        new_list = []
        for item in response_body:
            item = fixJSON(item)
            new_list.append(item)
        response_body = new_list
    return response_body

def checkTableExistence(data_pool, table):
    double_import = False
    data_source_id = table.data_source_id
    
    cc_data_sources = data_pool.get_data_connections()
    
    if sourceDataPoolID != targetDataPoolID and data_source_id != None: #if source data connection would need to be imported to target and not global scope
        try:
            cc_data_source = [cc_ds for cc_ds in cc_data_sources if cc_ds.id == data_source_id][0] #find data connection
        except Exception as e:
            raise Exception('ERROR: Data Source', data_source_id, 'cannot be found in source data pool.', 'Error:', e) 

        if cc_data_source.type_ == 'imported': #if data connection is imported
            double_import = True
            log_message(f'WARNING: Data Connection for table "{table.name}" is imported and cannot be imported again to target data pool. Attempting to find table in source data connection...')
            
    if double_import == False: #if non imported data connection
        if workspace_name == data_pool.name:
            data_connection_name = ''
        else:
            data_connection_name = data_pool.name + '_'
        if data_source_id == None: #source id is None when it is global
            data_connection_name = data_connection_name + 'global' 
        else:
            data_connection_name = data_connection_name + data_pool.get_data_connection(data_source_id).name
            
        final_table = table

    else: # if imported data connection
        export_data_pool_name = cc_data_source.name.split('_')[0] # imported data connection name is usually DataPoolName_DataConnectionName
        export_data_connection_name = cc_data_source.name.split('_')[-1]
        try: # try to find data pool
            export_data_pool = [export_data_pool for export_data_pool in celonis.data_integration.get_data_pools() if export_data_pool.name == export_data_pool_name][0]
        except Exception as e:
            raise Exception(f'ERROR: Export Data Pool "{export_data_pool_name}" cannot be found. Error: {e}') 
            
        if export_data_connection_name == 'global': #if it is global scope set source id to none
            export_data_source_id = None
        else: # if not global then find the data connection id using the name
            try:
                export_data_connection = [export_data_connection for export_data_connection in export_data_pool.get_data_connections() if export_data_connection.name == export_data_connection_name][0]
            except: 
                raise Exception(f'ERROR: Export Data Connection "{export_data_connection_name}" cannot be found in Data Pool "{export_data_pool_name}". Error: {e}') 

            export_data_source_id = export_data_connection.id
        
        try: # now try to find table with same name in the exported data connection
            export_table = [export_table for export_table in export_data_pool.get_tables() if export_table.data_source_id == export_data_source_id and export_table.name == table.name][0]
            final_table = export_table
            
            data_connection_name = export_data_pool.name + '_'
            if export_data_source_id == None: #source id is None when it is global
                data_connection_name = data_connection_name + 'global' 
            else:
                data_connection_name = data_connection_name + export_data_connection.name

            log_message(f'INFO: Identified Table "{export_table.name}" in source data connection "{data_connection_name}". Please note this table may not be identical to it\'s imported version.')
            
        except Exception as e:
            input_text = f'WARNING: Table "{table.name}" cannot be found in Export Data Pool "{export_data_pool_name}" Data Connection "{export_data_connection_name}". It is likely this table was created directly in the imported connection and does not exist in the original. Error: {e}'
            input_text += f'\n\nWould you like to create a view of the "{table.name}" table in the global scope of the "{data_pool.name}" data pool?'
            
            input_response = input(input_text)
            if input_response.lower() in ('y', 'yes'):
                try:
                    bootstrapper_data_job = [job for job in data_pool.get_jobs() if job.name == '000. OCPM Bootstrapper: Create Views'][0]
                    log_message(f'INFO: Global data job "000. OCPM Bootstrapper: Create Views" already exists. Attempting to create new transformation...')
                except:
                    bootstrapper_data_job = data_pool.create_job("000. OCPM Bootstrapper: Create Views")
                    (f'INFO: Global data job "000. OCPM Bootstrapper: Create Views" successfully created.')
                
                bootstrapper_job_transforms = bootstrapper_data_job.get_transformations()
                if len(bootstrapper_job_transforms) == 0:
                    first_transform = True
                    bootstrapper_task = bootstrapper_data_job.create_transformation(
                        name=f'Create Global View: "PLACEHOLDER_NAME"',
                        description=f'Creates a view in the global scope using table "{table.name}" from the "{cc_data_source.name}" scope.'
                    )
                else:
                    first_transform = False
                    bootstrapper_task = bootstrapper_job_transforms[0]
                    
                try:
                    bootstrapper_task.sync()
                    schema_param = [task_var.placeholder for task_var in bootstrapper_task.get_task_variables() if task_var.name == cc_data_source.name][0]
                    schema_param = schema_param.replace('DATASOURCE:', '')
                except Exception as e:
                    raise Exception(f'ERROR: Parameter in global scope for data connection "{cc_data_source.name}" cannot be found. Error: {e}')
                
                table_param = table.name.replace(' ', '_')
                final_table_name = f"{schema_param}_{table_param}_OCPM_Bootstrapper"
                
                if first_transform == False:
                    try:
                        bootstrapper_task = [task for task in bootstrapper_job_transforms if task.name == f'Create Global View: "{final_table_name}"'][0]
                        log_message(f'INFO: Global transformation to create view for table "{table.name}" in the "{cc_data_source.name}" scope already exists as " Create Global View: "{final_table_name}" ". Attempting to re-execute...')
                        new_transform = False
                    except Exception as e:
                        new_transform = True
                        bootstrapper_task = bootstrapper_data_job.create_transformation(
                            name=f'Create Global View: "{final_table_name}"',
                            description=f'Creates a view in the global scope using table "{table.name}" from the "{cc_data_source.name}" scope.'
                        )
                        
                else:
                    new_transform = True
                    bootstrapper_task.name = f'Create Global View: "{final_table_name}"'
                    bootstrapper_task.update()
                    
                if new_transform == True:
                    bootstrapper_task.update_statement(f"""CREATE OR REPLACE VIEW "{final_table_name}" AS (\n\tSELECT * FROM <%=DATASOURCE:{schema_param}%>."{table.name}"\n);""")
                    log_message(f'INFO: Global transformation " Create Global View: "{final_table_name}" " to create view for table "{table.name}" in the "{cc_data_source.name}" scope successfully created. Attempting to execute...')
                    
                    
                hyperlink = f'{celonis.client.base_url}/integration/ui/pools/{data_pool.id}/data-configuration/data-jobs/{bootstrapper_data_job.id}/transformations/{bootstrapper_task.id}?tab=transformation'
                    
                bootstrapper_task.enable()
                try:
                    bootstrapper_data_job.execute(transformation_ids = [bootstrapper_task.id] , wait = True)
                    log_message(f'INFO: Global transformation " Create Global View: "{final_table_name}" " successfully executed.')
                    bootstrapper_task.disable()
                    input('Please resynchronize the data connection to allow the new view to appear in the target data pool. Press enter once complete.')
                except Exception as e:
                    bootstrapper_task.disable()
                    raise Exception(f'ERROR: Global transformation " Create Global View: "{final_table_name}" " could not be executed. Please investigate here: {hyperlink}\nError: {e}')
                    
                try:
                    final_table = [final_table for final_table in data_pool.get_tables() if final_table.data_source_id == None and final_table.name == final_table_name][0]
                except Exception as e:
                    raise Exception(f'ERROR: Export Data Pool "{export_data_pool_name}" cannot be found. Error: {e}')
                    
                data_connection_name = data_pool.name + '_global'
                    
            else:
                raise Exception('Please see the documentation for alternative solutions.') ## DK NOTE 2025/02/27 -- We could hypothetically save the transformation with the default data source selected but without validation until the alternative is created
            

    data_sources = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_execution_factories_data_sources(celonis.client, workspace_id) 

    #check if data source exists in OCPM data pool
    if data_connection_name == 'global':
        data_source_id = ''
    else:
        try:
            data_source = [data_source for data_source in data_sources if data_connection_name == data_source.display_name][0] 
            data_source_id = data_source.data_source_id 
            data_source_type = data_source.data_source_type
        except Exception as e:
            raise Exception(f'ERROR: Data Source "{data_connection_name}" cannot be found. Please add it to the OCPM data model connections. Error: {e}') 
    
    available_tables = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_execution_factories_data_source_tables(celonis.client, workspace_id, data_source_id) #,  pagination={"requestMode": "ALL"}) 
    
    #check if table exists and retrieve columns
    try:
        ocpm_table = [ocpm_table for ocpm_table in available_tables if ocpm_table.name.name == table.name][0]   
        return data_source_id, ocpm_table.columns, final_table
    except Exception as e:
        raise Exception(f'ERROR: Table "{table.name}" cannot be found in the data connection "{data_connection_name}". Please resynchronize the data connection. Error: {e}')
        
    

def sanitizeName(name, name_type):
    #cleanedName = re.sub(string = name, repl = '', pattern = r'[\W]') #underscore causes issues?
    if name_type == 'table':
        length_limit = 40
    if name_type == 'column':
        length_limit = 50
        
    def fixNameNumbers(name):
        numberStart = re.findall('^[0-9]+[a-z]*', name)
        if len(numberStart) > 0:
            new_name = re.findall('^[0-9]+[a-z]*(.+)', name)[0] + numberStart[0]
            return new_name
        else:
            return name
            
            
    def cleanName(name):
        cleanedName = ''
        upper = True
        i = 0
        for char in name:
            i+= 1
            if char == '_' or char == ' ':
                upper = True
                cleanedName += char
                continue

            if upper == True or i == 1:
                try:
                    cleanedName += char.upper()
                except:
                    cleanedName += char
                upper = False
                continue

            if upper == False and i != 1:
                try:
                    cleanedName += char.lower()
                except:
                    cleanedName += char  
                continue

        cleanedName = re.sub(string = cleanedName, repl = '', pattern = '[^A-Za-z0-9]') 
        return cleanedName
    
    def shortenName(original_name, current_name, length_limit):
        while len(re.sub(string = current_name, repl = '', pattern = '[^A-Za-z0-9]') ) > length_limit:
            #print('Modified', name_type,'name', re.sub(string = current_name, repl = '', pattern = '[^A-Za-z0-9]'), 'for',original_name, 'is longer than limit of',length_limit, 'characters. Attempting to reduce.')
            
            current_length = len(re.sub(string = current_name, repl = '', pattern = '[^A-Za-z0-9]') )
            
                ### Try removing vowels
            new_name = ''
            remove_vowels = True
            for char in current_name:
                if char in 'aeiou' and remove_vowels == True:
                    remove_vowels = False
                else:
                    new_name += char
            current_name = new_name
            
            new_length = len(re.sub(string = current_name, repl = '', pattern = '[^A-Za-z0-9]') )
            
            if current_length == new_length:
                
                split_name = name.split('_')

                if len(split_name) > 1:
                    ### Try removing split parts
                    new_name = ''
                    remove_splits = True
                    for part in split_name:
                        if remove_splits == True and len(part) > 1 and len(part) < 4 and len(part) == len(re.sub(string = part, repl = '', pattern = '[0-9]')):
                            remove_splits = False
                        else:
                            new_name += '_' + part
                                                                                       
                    current_name = new_name[1:]
                    
                current_name = current_name
                new_length = len(re.sub(string = current_name, repl = '', pattern = '[^A-Za-z0-9]') )
                    
            if current_length == new_length:
                ### Try truncating
                current_name = re.sub(string = current_name, repl = '', pattern = '[^A-Za-z0-9]')[:length_limit]
                
        return current_name
                
    
                    
    cleanedName = cleanName(name)
    
    cleanedName = shortenName(name, cleanedName, length_limit)
    
    cleanedName = fixNameNumbers(cleanedName)
    
    if cleanedName != name:
        #print('Name of ',name_type, name, ' has been changed to ',cleanedName)
        pass
        
    return cleanedName

def check_rename(rename_raw, df, name_type):
    correct_rename = False

    while correct_rename == False:

        def modify_rename(rename_raw, name_type):
            modification_correct = False
            rename_sanitized = sanitizeName(rename_raw, name_type)
            while modification_correct == False:
                if rename_sanitized.lower() != rename_raw.lower():
                    keep_modification = input(f'Input name "{rename_raw}" does not adhere to required syntax and has been modified to "{rename_sanitized}". Type 1 to accept modification or try a different rename.')
                    if str(keep_modification).strip() == '1':
                        modification_correct = True
                        return rename_sanitized
                    else:
                        keep_modification = modify_rename(keep_modification, name_type)
                        return keep_modification
                else:
                    modification_correct = True
                    return rename_raw

        rename = modify_rename(rename_raw, name_type)

        if len(df.query(f'nameSanitizedIgnoreCase == "{rename.lower()}"').index) > 0:
            rename = input(f'Rename "{rename}" would create a duplicate name. Please try a different rename.')
            rename, df = check_rename(rename, df, name_type)
            correct_rename = True 
            return rename, df
        elif rename.lower() in ('id','time','epoch','activity') and name_type == 'column':
            rename = input(f'Rename "{rename}" is a reserved field. Please try a different rename.')
            rename, df = check_rename(rename, df, name_type)
            correct_rename = True 
            return rename, df
        else:
            correct_rename = True 
            return rename, df

def removeDupes(df, name_type):
    if name_type == 'table':
        entitiy = 'object'
    elif name_type == 'column':
        entity = 'attribute'
                
    values = df['nameSanitizedIgnoreCase'].value_counts()
        
    for duplicate in range(len(pd.DataFrame(values).query('count > 1').index)):
        dupe_name = pd.DataFrame(values).query('count > 1').iloc[duplicate].name
        dupe_count = pd.DataFrame(values).query('count > 1').iloc[duplicate]['count']
        
        input_text = f'\nWARNING: The following {dupe_count} {name_type}s are all being converted to {entity} name "{dupe_name}":\n'
        
        df_filtered = df.query(f'nameSanitizedIgnoreCase == "{dupe_name}"')
        for index in range(len(df_filtered)):
            if name_type == 'table':
                input_text += f"""\n\t{index + 1}. Table "{df_filtered.iloc[index][f'tableName']}" with alias "{df_filtered.iloc[index]['tableAlias']}"   """
            elif name_type == 'column':
                input_text += f"""\n\t{index + 1}. Column "{df_filtered.iloc[index][f'tableName']}"."{df_filtered.iloc[index][f'columnName']}"   """
            
        input_text += f'\n\nType -1 to exclude all {name_type}s.\nType 0 to overwrite all names.\nType the number of the {name_type} in the list above to select a {name_type} to keep the name "{dupe_name}".\n'
            
        input_response = input(input_text)
        
        correct_input = False
        
        while correct_input == False:
            try: 
                input_int = int(input_response)
                
                if input_int == -1:
                    correct_input = True
                    for ignore_index in df[(df.nameSanitizedIgnoreCase==f'{dupe_name}')].index:
                        df.at[ ignore_index, 'ignore' ] = True
                    
                    pass
    
                    
                elif input_int >= 0 and input_int <= dupe_count:
                    correct_input = True
                    ignore_others = False
                    if input_int != 0:
                        input_response = input(f'Would you like to exclude all other {name_type}s in list above?')
                        if input_response.lower() in ('y', 'yes'):
                            ignore_others = True
                                
                    for index in range(len(df_filtered)):
                        if name_type == 'table':
                            table_id = df_filtered.iloc[index]['tableId']
                            table_name = df_filtered.iloc[index]['tableName']
                            table_alias = df_filtered.iloc[index]['tableAlias']
                        elif name_type == 'column':
                            column_name = df_filtered.iloc[index]['columnName']
                        
                        if index + 1 == input_int:
                            continue
                        else: 
                            if ignore_others == True:
                                if name_type == 'table':
                                    df.at[ df[(df.tableId==f'{table_id}')].index[0], 'ignore' ] = True
                                elif name_type == 'column':
                                    df.at[ df[(df.columnName==f'{column_name}')].index[0], 'ignore' ] = True
                            else:
                                if name_type == 'table':
                                    rename_raw = input(f'What would you like to rename table "{table_name}" with alias "{table_alias}" to?')
                                elif name_type == 'column':
                                    rename_raw = input(f'What would you like to rename column "{column_name}" to?')
                                        
                                rename, df = check_rename(rename_raw, df, name_type)   
                                
                                if name_type == 'table':
                                    df.at[ df[(df.tableId==f'{table_id}')].index[0], 'nameSanitized' ] = rename
                                    df.at[ df[(df.tableId==f'{table_id}')].index[0], 'nameSanitizedIgnoreCase' ] = rename.lower()
                                elif name_type == 'column':
                                    df.at[ df[(df.columnName==f'{column_name}')].index[0], 'nameSanitized' ] = rename
                                    df.at[ df[(df.columnName==f'{column_name}')].index[0], 'nameSanitizedIgnoreCase' ] = rename.lower()

                    pass
                
                else:
                    raise Exception()
                
            except:
                input_response = input(f'Response "{input_response}" is not recognized. Please select one of the options listed above.')
                
    return df

def resolveReserved(df):
    names = df['nameSanitizedIgnoreCase']
    i = -1
    for name in names:
        i += 1
        if name in ('id','time','epoch','activity'):
            input_response = input(f'Attribute name "{name}" is reserved. Please type -1 to ignore this field. Otherwise, please provide a rename.')
            correct_input = False
            
            while correct_input == False:
                if str(input_response).strip() == '-1':
                    df.at[i, 'ignore']  = True
                    correct_input = True
                else:
                    rename, df = check_rename(input_response, df, name_type = 'column')   
                    df.at[i, 'nameSanitized']  = rename
                    df.at[i, 'nameSanitizedIgnoreCase']  = rename.lower()
                    correct_input = True
                    
    return df
                    

def create_fields(data_model, table_id, available_columns):
    #assemble column mapping
    
    table = data_model.get_table(table_id)
    table_name = table.name
    
    table_columns = table.get_columns()
    
    fields = [{'name': 'ID', 'dataType': 'CT_UTF8_STRING', 'namespace': 'custom', 'sql_formula': '', 'column_sql': ''}] 
    
    attributes_df = pd.DataFrame(columns = ['tableName','columnName', 'nameSanitized', 'nameSanitizedIgnoreCase' , 'ignore'])

    for column in available_columns:
        attributes_df.loc[len(attributes_df)] = [table_name, column.column_name, sanitizeName(column.column_name, 'column'), sanitizeName(column.column_name, 'column').lower(), False]
    
    attributes_df = removeDupes(df = attributes_df, name_type = 'column')
    attributes_df = resolveReserved(df = attributes_df)

    
    for column in [column for column in available_columns if column.column_name.lower()]: #DK Update
        
        try:
            cc_column = [cc_column for cc_column in table_columns if cc_column.name == column.column_name][0]
            
            try:
                cc_column_data_type = cc_column.type_.value
            except:
                cc_column_data_type = cc_column.type_
        except:
            if column.column_name.lower() == 'epoch':
                cc_column_data_type = 'INTEGER'
                
        
        ignore_column = attributes_df.query(f'columnName == "{column.column_name}"')['ignore'].iloc[0]
        column_name = attributes_df.query(f'columnName == "{column.column_name}"')['nameSanitized'].iloc[0]
        
        if ignore_column == False:
            
            try:
                column_data_type = column.type_.value
            except:
                column_data_type = column.type_

            try:
                timestamp_column = [pc.timestamp_column for pc in data_model.process_configurations if pc.activity_table_id == table_id][0]
            except:
                timestamp_column = None

            if column.column_name == timestamp_column:
                has_sorting = False
                try:
                    sorting_column = [pc.sorting_column for pc in data_model.process_configurations if pc.activity_table_id == table_id][0]
                except:
                    sorting_column = ''
                    
                if sorting_column != '':
                    
                    log_message(f'INFO: Identified sorting column "{sorting_column}". Attempting to leverage...')
                    
                    try:
                        cc_sorting_column = [cc_sorting_column for cc_sorting_column in table_columns if cc_sorting_column.name == sorting_column][0]

                        try:
                            cc_sorting_column_data_type = cc_sorting_column.type_.value
                        except:
                            cc_sorting_column_data_type = cc_sorting_column.type_
                    except:
                        if cc_sorting_column.name.lower() == 'epoch':
                            cc_sorting_column_data_type = 'INTEGER'
                        
                    if cc_sorting_column_data_type in ('INTEGER', 'FLOAT'):
                        has_sorting = True
                        sql_formula = f'\n\t,TIMESTAMPADD(microsecond, CAST("{table_name}"."{sorting_column}" AS INTEGER), CAST("{table_name}"."{column.column_name}" AS TIMESTAMP)) AS "Time"'
                        
                        if cc_sorting_column_data_type == 'FLOAT':
                            log_message(f'WARNING: Sorting column "{sorting_column}" is a float and will be rounded to sort timestamps. Process explorer activities with the same timestamp and rounded sorting value may not be sorted properly.')
                        
                    else:
                        log_message(f'WARNING: Sorting column "{sorting_column}" using non-numeric data type "{cc_sorting_column_data_type}" cannot be leveraged. Process explorer activities with the same timestamp may not be sorted properly.')
                    
                if has_sorting == False:
                    sql_formula = f'\n\t,CAST("{table_name}"."{column.column_name}" AS TIMESTAMP) AS "Time"'
                field = {
                    'name': 'Time', 
                    'dataType': column_data_type, 
                    'namespace': 'custom' ,
                    'sql_formula': sql_formula,
                    'column_sql': f'"{table_name}"."{column.column_name}"'
                }
                
            else: 

                if column_data_type == 'CT_INSTANT' or cc_column_data_type == 'DATE':
                    sql_formula = f'\n\t,CAST("{table_name}"."{column.column_name}" AS TIMESTAMP) AS "{column_name}"'
                    field_data_type = 'CT_INSTANT'
                elif column_data_type == 'CT_UTF8_STRING' and cc_column_data_type != 'DATE':
                    sql_formula = f'\n\t,CAST("{table_name}"."{column.column_name}" AS VARCHAR(255)) AS "{column_name}"'
                    field_data_type = column_data_type
                else:
                    sql_formula = f'\n\t,"{table_name}"."{column.column_name}" AS "{column_name}"'
                    field_data_type = column_data_type
                field = {
                    'name': column_name, 
                    'dataType': field_data_type, 
                    'namespace': 'custom' ,
                    'sql_formula': sql_formula,
                    'column_sql': f'"{table_name}"."{column.column_name}"'
                }
                
            fields.append(field)
        
    return fields 

def createPrimaryKey(data_model, table, entity_name, available_columns):
    column_df = pd.DataFrame([{'column_name': ac.column_name} for ac in available_columns])

    pks_df = pd.DataFrame(columns = ['entity_name', 'foreign_key_id', 'source_table_id', 'source_table_name', 'source_table_alias', 'pk_columns', 'pk_formula', 'target_table_id', 'target_table_name', 'target_table_alias'])
    pk_index = 0
    for fk in data_model.foreign_keys:
        if fk.source_table_id == table.id:
            pk_index += 1
            pk_df = pd.DataFrame(columns = ['column_name'])
            for col in fk.columns:
                col_df_filtered = column_df.query(f'column_name == "{col.source_column_name}"')
                pk_df = pd.concat([pk_df, col_df_filtered])
            pk_df.sort_index(inplace=True)
            pk_id = ''
            pk_form = ''
            for pk_col in pk_df['column_name']:
                pk_id += f'"{table.name}"."{pk_col}" || '
                pk_form += f'CAST("{table.name}"."{pk_col}" AS VARCHAR(255)) || '
            pk_id = pk_id[0:-4]
            pk_form = pk_form[0:-4]

            target_table = data_model.get_table(fk.target_table_id)
            pk_dict = {'entity_name': entity_name, 'foreign_key_id': fk.id, 'source_table_id': table.id, 'source_table_name': table.name, 'source_table_alias': table.alias, 'pk_columns': pk_id, 'pk_formula': pk_form, 'target_table_id': target_table.id, 'target_table_name': target_table.name, 'target_table_alias': target_table.alias}
            pk_df = pd.DataFrame(data = pk_dict, index = [pk_index])
            pks_df = pd.concat([pks_df, pk_df])

    pk_columns = pks_df['pk_columns'].unique()
    
    pk_formulas = pks_df['pk_formula'].unique()

    if len(pk_formulas) > 1:
        if table.alias != None and table.alias != '':
            source_alias_text = f' with alias "{table.alias}"'
        else:
            source_alias_text = ''

        input_text = f'WARNING: Found multiple potential primary keys for table "{table.name}"{source_alias_text}:'

        pk_index = 0
        for pk_column in pk_columns:
            pk_index += 1
            pks_df_filtered = pks_df.query(f"pk_columns == '{pk_column}'")

            input_text += f'\n\n\t{pk_index}. {pk_column}\n'

            for filtered_pk_index in range(len(pks_df_filtered.index)):
                filtered_pk = pks_df_filtered.iloc[filtered_pk_index]

                target_table_name = filtered_pk['target_table_name']
                target_table_alias = filtered_pk['target_table_alias']

                if target_table_alias != None and target_table_alias != '':
                    target_alias_text = f' with alias "{target_table_alias}"'
                else:
                    target_alias_text = ''

                input_text += f'\n\t\tTable "{target_table_name}"{target_alias_text}'

        input_text += f'\n\nType -1 to skip primary key creation.\nType 0 to create new primary key.\nType the number of the formula in the list above to select a primary key.\n\n'

        input_response = input(input_text)
        correct_input = False

        while correct_input == False:
            try:

                try:
                    input_response = int(input_response)
                except:
                    raise Exception('Must be an integer.\n\n')

                if input_response < 0:
                    raise Exception(f'"{input_response}" is a negative integer.\n\n')

                elif input_response == -1:
                    correct_input = True
                    final_pk_formula = '\'<%=CHANGE_PRIMARY_KEY%>\''

                elif input_response == 0:
                    correct_input = True
                    final_pk_formula = None

                elif input_response > 0 and input_response <= len(pk_formulas):
                    correct_input = True
                    final_pk_formula = pk_formulas[input_response - 1]

                elif input_response > len(pk_formulas):
                    raise Exception(f'"{input_response}" does not correspond to formula in list.\n\n')

            except Exception as e:
                new_input = f'\nResponse "{input_response}" is not recognized. Please configure response based on the options listed above. {e}'
                correct_response = False
                input_response = input(new_input)

        input_text = f'WARNING: The existence of multiple potential primary keys may indicate incorrectly defined join directionality. Please check the join definitions for this table.\n\n'
        input_text += f'Type 0 to ignore.\nType the numbers of the formulas in the list above, separated by commas, to switch the cardinality of any of the data model joins.\n\n'

        input_responses = input(input_text)
        correct_response = False

        while correct_response == False:
            skip = False
            input_responses_list = input_responses.strip().split(',')
            update_pks = []
            try:
                for pk_response in input_responses_list:
                    try:
                        pk_response = int(pk_response)
                    except Exception as e:
                        raise Exception(f'Must be a list of integers.\n\n')

                    if pk_response > len(pk_formulas):
                        raise Exception(f'"{pk_response}" does not correspond to field in list.\n\n')

                    elif pk_response < 0:
                        raise Exception(f'"{pk_response}" is a negative integer.\n\n')

                    elif len(input_responses_list) > 1 and pk_response == 0:
                        raise Exception(f'"{pk_response}" cannot be used with other values.\n\n')

                    elif len(input_responses_list) == 1 and pk_response == 0:
                        correct_response = True
                        skip = True

                    elif input_response == pk_response:
                        raise Exception(f'"{pk_response}" is the assigned primary key and joins using it cannot be altered.\n\n')

                    elif pk_response > 0 and pk_response <= len(pk_formulas) and pk_response != input_response:
                        update_pks.append(pk_formulas[pk_response - 1])

                if skip == False:
                    correct_response = True
                    for update_pk in update_pks:

                        pks_df_filtered = pks_df.query(f"pk_formula == '{update_pk}'")

                        for filtered_pk_index in range(len(pks_df_filtered.index)):
                            filtered_pk = pks_df_filtered.iloc[filtered_pk_index]

                            foreign_key_id = filtered_pk['foreign_key_id']

                            target_table_name = filtered_pk['target_table_name']
                            target_table_alias = filtered_pk['target_table_alias']

                            if target_table_alias != None and target_table_alias != '':
                                target_alias_text = f' with alias "{target_table_alias}"'
                            else:
                                target_alias_text = ''


                            try:
                                fk = data_model.get_foreign_key(foreign_key_id)
                                fk_dict = fk.json_dict()
                                fk_columns = []

                                source_tab = fk_dict['source_table_id']
                                target_tab = fk_dict['target_table_id']

                                fk_dict['source_table_id'] = target_tab
                                fk_dict['target_table_id'] = source_tab

                                for fk_col in fk_dict['columns']:
                                    source_col = fk_col['source_column_name']
                                    target_col = fk_col['target_column_name']

                                    fk_col['source_column_name'] = target_col
                                    fk_col['target_column_name'] = source_col

                                    fk_columns.append(fk_col)

                                fk_dict['columns'] = fk_columns

                                celonis.client.request(url = f'{celonis.client.base_url}/integration/api/pools/{data_model.data_pool_id}/data-models/{data_model.id}/foreign-keys/{foreign_key_id}', method = 'PUT', request_body = fixJSON(fk_dict))
                                log_message(f'INFO: Foreign key cardinality between "{table.name}"{source_alias_text} and table "{target_table_name}"{target_alias_text} successfully switched.')
                            except Exception as e:
                                log_message(f'ERROR: Foreign key cardinality between "{table.name}"{source_alias_text} and table "{target_table_name}"{target_alias_text} could not be switched. Error: {e}')

            except Exception as e:
                pk_input = f'\nResponse "{str(input_responses)}" is not recognized. Please configure response based on the options listed above. {e}'
                correct_response = False
                input_responses = input(pk_input)


    elif len(pk_formulas) == 1:
        final_pk_formula = pk_formulas[0]

    elif len(pk_formulas) == 0:
        final_pk_formula = None
        
    return final_pk_formula

def createForeignKey(fk, source_table, target_table):
    column_df = pd.DataFrame([{'source_column_name': column.name} for column in source_table.get_columns()])
    pk_index = 0

    fk_df = pd.DataFrame(columns = ['source_column_name', 'target_column_name'])
    for col in fk.columns:
        col_df_filtered = column_df.query(f'source_column_name == "{col.source_column_name}"')
        col_df_filtered = col_df_filtered.assign(target_column_name=col.target_column_name)
        
        fk_df = pd.concat([fk_df, col_df_filtered])
    fk_df.sort_index(inplace=True)
    
    fk_form = ''
    for fk_col in fk_df['target_column_name']:
        fk_form += f'CAST("{target_table.name}"."{fk_col}" AS VARCHAR(255)) || '
    fk_form = fk_form[0:-4]
            
    return fk_form

def assembleSQL(data_model, table_id, final_table_name, entity_name, available_columns, isActivityTable, case_tables, activity_tables, fields):  
    
    #assemble primary key and foreign key mapping (this is awful - rewrite it)
    table = data_model.get_table(table_id)
    table_name = table.name
    if table.alias != None and table.alias != '':
        object_name_raw = sanitizeName(table.alias, 'table')
    else:
        object_name_raw = sanitizeName(table.name, 'table')
    
    primary_key_columns = []
    sql_statement = 'SELECT\n\t\'<%=CHANGE_PRIMARY_KEY%>\' AS "ID"' #changed this to a string so that it works in sql validation
    sql_joins = '' #creating empty join string
    relationship_table_sqls = []
    
    primaryKey = createPrimaryKey(data_model, table, entity_name, available_columns)
    
    joinedActivityTable = None
    join_count = 0
    if not isActivityTable:
        
        if primaryKey != None: #if no primary key yet
            sql_statement = f'SELECT\n\t{primaryKey} AS "ID"'
        
        for fk in data_model.foreign_keys:
            if fk.source_table_id == table_id: #source table is the 1 in a 1:N relationship (think dimension table)
                table_name = data_model.get_table(table_id).name
                source_object_name_raw = [table.name if table.alias == None or table.alias == '' else table.alias for table in [data_model.get_table(table_id)]][0]
                
                if fk.target_table_id in activity_tables:
                    joinedActivityTable = 'FACT_TABLE'
                    
                    try:
                        case_table_id = [case_tables[case_table] for case_table in range(len(case_tables)) if activity_tables[case_table] == fk.target_table_id][0] 
                    except:
                        case_table = None
                        
                    if case_table_id == None:
                        log_message(f'WARNING: Object "{entity_name}" table "{table_name}" is joined to Activity Table "{table_name}" with no assigned case table.') # object may not be connected if only connection is through activity table
                    else:
                        join_count += 1
                        case_table = data_model.get_table(case_table_id)
                        if case_table.alias != None and case_table.alias != '':
                            case_table_object_name = case_table.alias
                        else:
                            case_table_object_name = case_table.name
                        
                        case_table_object_name = sanitizeName(case_table_object_name, 'table')
                        case_column = [process_config.case_id_column for process_config in data_model.process_configurations if process_config.activity_table_id == fk.target_table_id][0]

                        if len(fk.columns) == 1 and fk.columns[0].target_column_name == case_column: #if one join to activity table and the join is a 1:N to the case table column -- update to left join and use activity columns to link to case table? since case id does not always result in primary key -- should revisit this logic maybe?  -- DK NOTE 2024/11/18
                            joinedActivityTable = 'CASE_ID'
                            if case_table_id != fk.source_table_id:
                                foreignCaseKey = f'"{table_name}"."{fk.columns[0].source_column_name}"'
                                sql_statement += f'\n\t,CAST({foreignCaseKey} AS VARCHAR(255)) AS "{case_table_object_name}"'
                        else:
                            log_message(f'INFO: Object "{entity_name}" using table "{table_name}" has indirect M:N relationship with Object "{case_table_object_name}" using "{case_table.name}" through its activity table. Creating relationship SQL statement...')
                            activity_table = data_model.get_table(fk.target_table_id) # fk table arg?
                        
#                         if activity_table.alias != None and activity_table.alias != '':
#                             activity_table_object_name = activity_table.alias
#                         else:
#                             case_table_object_name = activity_table.name
#                         activity_table_name = data_model.get_table(fk.target_table_id).name
                        
                            relationship_sql = ''

                            foreignID = createForeignKey(fk = fk, source_table = table, target_table = activity_table)
                            
                            source_fk = [sfk for sfk in data_model.foreign_keys if sfk.source_table_id == case_table.id and sfk.target_table_id == fk.target_table_id][0] # fk fk arg
                            sourceID = createForeignKey(fk = source_fk, source_table = case_table, target_table = activity_table)
                            
                            relationship_sql += f'SELECT\n\t{foreignID} AS "ID"'      
                            relationship_sql += f'\n\t,{sourceID} AS "{case_table_object_name}"'
                            relationship_sql += f'FROM "{activity_table.name}"'

                            relationship_table = {
                                'source_object': entity_name,
                                'target_object': sanitizeName(case_table_object_name, 'table'),
                                'activity_table': activity_table.name, ##FIX MAYBE??
                                'sql': relationship_sql
                            }

                            relationship_table_sqls.append(relationship_table)
                            
                else:
                    join_count += 1
                        
                
        for fk in data_model.foreign_keys:      #### get relationships to other fields 
            if fk.target_table_id == table_id: 
                table_name = data_model.get_table(table_id).name
                target_object_name_raw = [table.name if table.alias == None or table.alias == '' else table.alias for table in [data_model.get_table(table_id)]][0]
                if fk.source_table_id in activity_tables: #if joined to activity table!!!
                    joinedActivityTable = 'DIMENSION_TABLE'
                    
                    try:
                        case_table_id = [case_tables[case_table] for case_table in range(len(case_tables)) if activity_tables[case_table] == fk.source_table_id][0]
                    except:
                        case_table_id = None
                        
                    if case_table_id == None:
                        log_message(f'WARNING: Object "{entity_name}" table "{table_name}" is joined to Activity Table "{table_name}" with no assigned case table.')
                    else:
                        join_count += 1
                        case_table = data_model.get_table(case_table_id)

                        if case_table.alias != None and case_table.alias != '':
                            case_table_object_name = case_table.alias
                        else:
                            case_table_object_name = case_table.name
                        
                        case_table_object_name = sanitizeName(case_table_object_name, 'table')
                        case_column = [process_config.case_id_column for process_config in data_model.process_configurations if process_config.activity_table_id == fk.source_table_id][0]

                        if len(fk.columns) == 1 and fk.columns[0].source_column_name == case_column: #if one join to activity table and the join is a 1:N to the case table column -- revisit this logic maybe -- DK NOTE 2024/11/18
                            joinedActivityTable = 'CASE_ID'
                            foreignCaseKey = f'"{table_name}"."{fk.columns[0].target_column_name}"'
                            sql_statement += f'\n\t,CAST({foreignCaseKey} AS VARCHAR(255)) AS "{case_table_object_name}"'

                        else:
                            log_message(f'INFO: Object "{entity_name}" using table "{table_name}" has indirect 1:N relationship with Object "{case_table_object_name}" using "{case_table.name}" through its activity table. Adding LEFT JOIN to SQL statement...') ## possibly missing the actual columns to join on? -- revisit this logic -- ## DK REORDER 24/12/6
                            activity_table = data_model.get_table(fk.source_table_id)
                            sql_joins += f'\nLEFT JOIN "{activity_table.name}" ON 1=1'
                            for key_pair in fk.columns:
                                sql_joins += f'\n\tAND "{table_name}"."{key_pair.target_column_name}" = "{activity_table.name}"."{key_pair.source_column_name}"'
                
                else: # if joined to normal table
                    join_count += 1
                    source_table = data_model.get_table(fk.source_table_id) # fk table arg

                    if source_table.alias != None and source_table.alias != '':
                        source_table_object_name = source_table.alias
                    else:
                        source_table_object_name = source_table.name
                            
                    foreignKey = createForeignKey(fk = fk, source_table = source_table, target_table = table)
                    source_table_object_name = sanitizeName(source_table_object_name, 'column') 
                    sql_statement += f'\n\t,CAST({foreignKey} AS VARCHAR(255)) AS "{source_table_object_name}"'
                    
        if join_count == 0:
            log_message(f'WARNING: Object "{entity_name}" using table "{table_name}" has no object relationships.')
        
    else: #only care about the case table
        object_name_raw = [table.name if table.alias == None or table.alias == '' else table.alias for table in [data_model.get_table(table_id)]][0]
        primaryKey = " || '_' || "
        process_configuration = [pc for pc in data_model.process_configurations if pc.activity_table_id == table_id][0]
        sql_statement = f'SELECT\n\t"{table_name}"."{process_configuration.activity_column}" {primaryKey} "{table_name}"."{process_configuration.case_id_column}" {primaryKey} "{table_name}"."{process_configuration.timestamp_column}" AS "ID"'
        #case_table = case_tables ## is just singular ID 
        
        for case_table in case_tables:
            case_table_object_name = case_table['ocName']
            
            case_table_data = data_model.get_table(case_table['tableId']) #fk table arg
            case_fk = [sfk for sfk in data_model.foreign_keys if sfk.source_table_id == case_table['tableId'] and sfk.target_table_id == table_id][0] # fk fk arg
            caseID = createForeignKey(fk = case_fk, source_table = case_table_data, target_table = table)
        
            sql_statement += f'\n\t,{caseID} AS "{case_table_object_name}"' #this assumes a case key is less than 255 characters -- will be problematic if not
            
    if primaryKey is None:
        pk_input = f''
        for field_index in range(len(fields)):
            if field_index != 0:
                column_sql = fields[field_index]['column_sql']
                pk_input += f"""\n\t{field_index}. {column_sql}   """
        
        pk_input += f'\n\nWARNING: No primary key found for object "{entity_name}" using table "{table_name}"\n\nType 0 to replace manually.\nType the numbers of the columns in the list above, separated by commas, to create a composite primary key.\n\n'
        
        pk_responses = input(pk_input)
        
        correct_response = False
        
        while correct_response == False:
            skip = False
            primary_key = ''
            pk_responses_list = pk_responses.strip().split(',')
            try:
                for pk_response in pk_responses_list:
                    try:
                        pk_response = int(pk_response)
                    except Exception as e:
                        raise Exception(f'Must be a list of integers.\n\n')

                    if pk_response > len(fields):
                        raise Exception(f'"{pk_response}" does not correspond to field in list.\n\n')
                        
                    elif pk_response < 0:
                        raise Exception(f'"{pk_response}" is a negative integer.\n\n')
                        
                    elif len(pk_responses_list) > 1 and pk_response == 0:
                        raise Exception(f'"{pk_response}" cannot be used with other values.\n\n')
                        
                    elif len(pk_responses_list) == 1 and pk_response == 0:
                        log_message(f'WARNING: Please update the SQL for the ID column of object "{entity_name}" using table "{table_name}".')
                        correct_response = True
                        skip = True
                    
                    elif pk_response > 0 and pk_response <= len(fields):
                        column_sql = fields[pk_response]['column_sql']
                        primary_key += f'CAST({column_sql} AS VARCHAR(255)) || '
                        primary_key_columns.append(column_sql.split('.')[1].strip('"'))
                    
                if skip == False:
                    primary_key = primary_key[:-4]
                    correct_response = True
                    try:
                        sql_statement = sql_statement.replace('\'<%=CHANGE_PRIMARY_KEY%>\'', primary_key) 
                    except Exception as e:
                        log_message(f'ERROR: SQL for the ID column of object "{entity_name}" using table "{table_name}" could not be updated to {primary_key}. Error: {e}')
                    
            except Exception as e:
                pk_input = f'\nResponse "{str(pk_responses)}" is not recognized. Please configure response based on the options listed above. {e}'
                correct_response = False
                pk_responses = input(pk_input)
        
    #assemble column mapping
    fields_final = []
    
    for field in fields:
        fields_final.append(
            {
                'name': field['name'],
                'dataType': field['dataType'],
                'namespace': field['namespace']
            }
        )
    
        sql_statement += field['sql_formula']

    sql_statement += f'\nFROM "{final_table_name}" AS "{table_name}"' + sql_joins
        
    return sql_statement, fields_final, relationship_table_sqls, primary_key_columns

def deleteObject(object_id = None, object_name = None):
    if object_name == None and object_id == None:
        raise Exception('Object ID or Name not specified')
    elif object_id != None:
        try:
            object_name = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_type_id = object_id).name
            
        except:
            raise Exception(f'Object with ID {object_id} cannot be found')
    elif object_name != None:
        try:
            object_id = [o.id for o in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content if o.name.lower() == object_name.lower()][0]
           
        except:
            raise Exception(f'Object with Name {object_name} cannot be found')
    
    ### Need to remove object from any perspectives it is in

    def find_related_objects(perspective, object_name):
        for object_data in perspective.json_dict()['objects']:
            if object_data['name'].lower() == object_name.lower():
                return object_name

    def update_perspective(perspective, object_name):
        updated_perspective_objects = []
        perspective_data = perspective.json_dict()
        for object_data in perspective_data['objects']:
            if object_data['name'].lower() != object_name.lower():
                updated_relationship_data = []
                for relationship in object_data['relationships']:
                    if relationship['name'].lower() != object_name.lower():
                        try:
                            del perspective_data['origin_ref']
                        except:
                            pass
                        updated_relationship_data.append(relationship)

                object_data['relationships'] = updated_relationship_data

                updated_perspective_objects.append(object_data)

        perspective_data['objects'] = updated_perspective_objects
        del perspective_data['change_date']
        del perspective_data['changed_by'] 
        del perspective_data['created_by']
        del perspective_data['creation_date']
        try:
            del perspective_data['default_projection']
        except:
            pass
        try:
            del perspective_data['base_ref']
        except:
            pass
        return perspective_data

    for perspective in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_perspectives(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content:
        
        object_in_perspective = find_related_objects(perspective, object_name)  
        if object_in_perspective != None:
            if object_in_perspective.lower() == object_name.lower():
                log_message(f'Object "{object_name}" used in Perspective "{perspective.name}". Attempting to update...')
                perspective_json = update_perspective(perspective, object_name)
                perspective_json = fixJSON(perspective_json)

                try:
                    BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_perspectives_perspective_id(celonis.client, workspace_id, perspective_id = perspective.id, request_body = perspective_json)
                    log_message(f'INFO: Successfully removed object "{object_name}" from Perspective "{perspective.name}".')
                except Exception as e:
                    log_message(f'ERROR: Failed to remove object "{object_name}" from Perspective "{perspective.name}". Deleting Perspective... Error: {e}')
                    try:
                        BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_perspectives_perspective_id(celonis.client, workspace_id, perspective.id)
                        log_message(f'INFO: Successfully deleted Perspective "{perspective.name}".')
                    except Exception as e:
                        log_message(f'ERROR: Failed to delete Perspective "{perspective.name}". Error: {e}')
    
    ### Need to delete transformations related to object
    transformations = [factory.factory_id for factory in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_factories_sql(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content if factory.target.kind == 'OBJECT' and factory.target.entity_ref.name.lower() == object_name.lower()]

    for factory_id in transformations:
        try:
            BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id)
            log_message(f'INFO: Transformation "{factory_id}" successfully deleted.')
        except Exception as e:
            log_message(f'ERROR: Transformation "{factory_id}" could not be deleted. Error: {e}')
    
    ### Need to delete relationships object has


    try:
        object_relationships = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects_object_type_id_relationships(celonis.client, workspace_id, object_id).content
        for relationship in object_relationships:
            if relationship.target.object.name.lower() == object_name.lower():
                try:
                    if relationship.owner == 'SOURCE':
                        owner_object_id = [source.id for source in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content if source.name.lower() == relationship.source.object.name.lower()][0]
                        relationship_name = relationship.source.relationship.name
                        
                    if relationship.owner == 'TARGET':
                        owner_object_id = object_id
                        relationship_name = relationship.target.relationship.name

                    BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_types_objects_object_type_id_relationships_target_type_id(celonis.client, workspace_id, object_type_id = owner_object_id, target_type_id = object_id, relationship = relationship_name)
                    log_message(f'INFO: Object relationship "{relationship.source.object.name}" to "{relationship.target.object.name}" with cardinality "{relationship.cardinality}" successfully deleted.')
                except Exception as e:
                    log_message(f'ERROR: Object relationship "{relationship.source.object.name}" to "{relationship.target.object.name}" with cardinality "{relationship.cardinality}" could not be deleted. Error: {e}')
    except:
        pass
    
    events = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_events(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content
    for event in events:
        event_id = event.id
        event_name = event.name
        event_data = event.json_dict()
        update_event = False
        event_relationships = event_data['relationships']
        new_event_relationships = []
        for event_relationship in event_relationships:
            event_relationship_cardinality = event_relationship['cardinality']
            if event_relationship['name'].lower() != object_name.lower() and event_relationship['target']['object_ref']['name'].lower() != object_name.lower():
                new_event_relationships.append(event_relationship)
            else:
                update_event = True
        
        if update_event == True:
            event_data['relationships'] = new_event_relationships
            del event_data['change_date']
            del event_data['changed_by']
            del event_data['created_by']
            del event_data['id']
            event_data = fixJSON(event_data)
            try:
                BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_types_events_event_type_id(celonis.client, workspace_id, event_type_id = event_id, request_body = event_data) 
                log_message(f'INFO: Event "{event_name}" with relationship to "{object_name}" successfully removed.')
            except Exception as e:
                log_message(f'ERROR: Event "{event_name}" with relationship to "{object_name}" could not be removed. Error: {e}')


    ### delete object 
    try:
        BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_id)
        log_message(f'INFO: Object "{object_name}" with ID "{object_id}" successfully deleted.')
    except Exception as e:
        log_message(f'ERROR: Object "{object_name}" with ID "{object_id}" could not be deleted. Error: {e}')
        
def createSQLMapping(entity_name, fields):
    sql_mapping = pd.DataFrame(columns = ['cc_table', 'cc_alias','cc_column','oc_table','oc_column'])
    for field in fields:
        try:
            column_sql = field['column_sql'][1:-1].split('"."')
            sql_mapping.loc[len(sql_mapping)] = [column_sql[0], '', column_sql[1], entity_name, field['name']]
        except: 
            continue
    return sql_mapping

def createObject(object_name, fields, objects_df):

    final_object_name = False
    original_object_name = object_name
    
    while final_object_name == False:
    
        request_body = {
            'name': object_name,
            'tags': [],
            'categories': [{'metadata': {'name': 'Processes', 'namespace': 'celonis'},
              'values': [{'displayName': 'Compatibility Mode',
                'name': 'CompatibilityMode',
                'namespace': 'custom'}]}],
            'relationships': [],
            'fields': fields,
            'color': '#0A18BB'}

        objects = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content

            ### DK UPDATED START 2024/10/30
        if object_name.lower() in [o.name.lower() for o in objects]:
            input_response = input(f'WARNING: Object "{object_name}" already exists. \n\n\tType -2 to ignore this object (Remove its use from other objects and events).\n\tType -1 to skip this object (Maintain its use in other objects and events).\n\tType 0 to merge with existing object.\n\tType 1 to replace existing object.\n\tType 2 to create a new object with a different name.\n')
            object_id = [o.id for o in objects if object_name.lower() == o.name.lower()][0]

            correct_input = False
            while correct_input == False:
                try:
                    input_response = int(input_response.strip()) #check integer
                    
                    if input_response == -2: # ignore -- continue -- need to mark to skip transformation
                        final_object_name = True
                        correct_input = True
                        return 'IGNORE', None, object_name, fields

                    if input_response == -1: # skip -- continue -- need to mark to skip transformation
                        final_object_name = True
                        correct_input = True
                        return 'SKIP', None, object_name, fields

                    elif input_response == 0: # merge -- more logic required here
                        correct_input = True
                        final_object_name = True
                        object_data = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_type_id = object_id)
                        object_data = fixJSON(json.loads(object_data.json()))
                        
                        try:
                            process_tags = [v['name'] for v in object_data['categories'][0]['values']]
                            if 'CompatibilityMode' not in process_tags:
                                object_data['categories'][0]['values'].append({'displayName': 'Compatibility Mode', 'name': 'CompatibilityMode', 'namespace': 'custom'})
                        except:
                            object_data['categories'] = [{}]
                            object_data['categories'][0]['metadata'] = {'name': 'Processes', 'namespace': 'celonis'}
                            object_data['categories'][0]['values'] = [{'displayName': 'Compatibility Mode', 'name': 'CompatibilityMode', 'namespace': 'custom'}]

                        try:
                            del object_data['id']
                        except:
                            pass

                        try:
                            del object_data['changeDate']
                        except:
                            pass

                        try:
                            del object_data['changedBy']
                        except:
                            pass

                        try:
                            del object_data['createdBy']
                        except:
                            pass

                        try:
                            del object_data['creationDate']
                        except:
                            pass
                        
                        fields_updated = []
                        for field in fields:
                            if field['name'].lower() not in [f['name'].lower() for f in object_data['fields']]:
                                object_data['fields'].append(field)
                                fields_updated.append(field)
                            else:
                                field_index = [i for i in range(len(object_data['fields'])) if object_data['fields'][i]['name'].lower() == field['name'].lower()][0]
                                current_field_data_type = object_data['fields'][field_index]['dataType']

                                new_field_data_type = field['dataType']
                                field_name = field['name']

                                if current_field_data_type != new_field_data_type:
                                    correct_field_input = False
                                    field_input = input(f'Attribute "{field_name}" already exists in object "{object_name}" with data type "{current_field_data_type}". \n\n\tType 0 to keep existing data type.\n\tType 1 to replace existing data type with "{new_field_data_type}".\n\tType 2 to create a new attribute with a different name.\n')

                                    while correct_field_input == False:
                                        try:
                                            field_input = int(field_input)

                                            if field_input == 0: ## do nothing -- might need to update how sql gets created 
                                                correct_field_input = True

                                            if field_input == 1: # replace object_data['fields']
                                                object_data['fields'][field_index]['dataType'] = new_field_data_type

                                                correct_field_input = True

                                            if field_input == 2: # request new name -- create dataframe of objects then use checkRename combined with removeDupes 


                                                fields_df = pd.DataFrame(columns = ['columnName', 'nameSanitized', 'nameSanitizedIgnoreCase' , 'ignore'])
                                                for df_field in object_data['fields']:
                                                    fields_df.loc[len(fields_df)] = ['', df_field['name'], df_field['name'].lower(), False]

                                                #fields_df.loc[len(fields_df)] = ['', field['name'], field['name'].lower(), False]

                                                rename_raw = input(f'What would you like to rename field "{field_name}" to?')

                                                rename, fields_df = check_rename(rename_raw, fields_df, name_type = 'column')   

                                                field['name'] = rename

                                                object_data['fields'].append(field)

                                                correct_field_input = True
                                                
                                            fields_updated.append(field)

                                        except:
                                            field_input = input(f'Response "{field_input}" is not recognized. Please select one of the options listed above.')
                                            
                                elif current_field_data_type == new_field_data_type: #dk added, add updated fields that exist in both with same data type
                                    fields_updated.append(field)

                        try:
                            entity = BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_type_id = object_id, request_body = object_data)
                            log_message(f'INFO: Object "{object_name}" successfully merged.')
                            return 'MERGE', entity, object_name, fields_updated
                        except Exception as e:
                            log_message(f'ERROR: Object "{object_name}" could not be merged. Error: {e}')
                            return 'FAILURE', None, object_name, fields_updated


                    elif input_response == 1: # replace -- run the put request 
                        correct_input = True
                        final_object_name = True

                        try:
                            deleteObject(object_id = object_id)

                            try:
                                entity = BusinessLandscapePycelonis.post_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id, request_body = request_body)
                                #BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_type_id = object_id, request_body = request_body)
                                log_message(f'INFO: Object "{object_name}" successfully replaced.')
                                return 'REPLACE', entity, object_name, fields
                            except Exception as e:
                                log_message(f'ERROR: Object "{object_name}" could not be replaced. Error: {e}')
                                return 'FAILURE', None, object_name, fields

                        except Exception as e:
                            log_message(f'ERROR: Object "{object_name}" could not be replaced. Error: {e}')
                            return 'FAILURE', None, object_name, fields

                    elif input_response == 2: # new object -- need to ask for new name and do checks as required  -- maybe pull in the objects df -- needed anyhow
                        correct_input = True

                        ## UPDATE
                        rename_raw = input(f'What would you like to rename object "{object_name}" to?')
                        object_name, objects_df = check_rename(rename_raw, objects_df, name_type = 'table')

                    else: #bad number
                        raise Exception()

                except:
                    input_response = input(f'Response "{input_response}" is not recognized. Please select one of the options listed above.')


            ### DK UPDATED END 2024/10/30

        else:
            final_object_name = True
            try:
                entity = BusinessLandscapePycelonis.post_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id, request_body = request_body)
                log_message(f'INFO: Object "{object_name}" successfully created.')
                return 'SUCCESS', entity, object_name, fields
            except Exception as e:
                log_message(f'ERROR: Object "{object_name}" could not be created. Error: {e}')
                return 'FAILURE', None, object_name, fields
            

def createTrafo(type_name, data_source_id, sql_statement, isActivityTable, create_type = 'REPLACE'):

    display_name = f'{type_name}_{str(datetime.now())}'
    
    if not isActivityTable:
        objects = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content 
        entity_ref = [o.name for o in objects if type_name.lower() == o.name.lower()][0]
    else:
        entity_ref = type_name
    
    #create empty trafo
    trafo_type = 'EVENT' if isActivityTable else 'OBJECT'
    request_body = {
        'description': '',
        'namespace': 'custom',
        'target': {'kind': trafo_type,
                   'entityRef': {'name': entity_ref, 'namespace': 'custom'}},
        'dataConnectionId': data_source_id,
        'displayName': display_name,
        'extensionSqlFactories': [],
        'localParameters': []}
    
    factories = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_factories_sql(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content 
    
    if create_type == 'REPLACE' or create_type == 'SUCCESS':
        if type_name.lower() in [factory.target.entity_ref.name.lower() for factory in factories]:
            log_message(f'WARNING: Transformation for {trafo_type} "{type_name}" already exists. Attempting to update...')
            factory_id = [f.factory_id for f in factories if type_name.lower() == f.target.entity_ref.name.lower()][0]
            try:
                entity = BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = factory_id, request_body = request_body)    # put back later
            except Exception as e:
                log_message(f'ERROR: Transformation for {trafo_type} "{type_name}" could not be updated. Error: {e}')  
                return 'FAILURE', None

        else:
            try:
                entity = BusinessLandscapePycelonis.post_api_v2_workspaces_workspace_id_factories_sql(celonis.client, workspace_id, request_body = request_body)    # put back later
            except Exception as e:
                log_message(f'ERROR: Transformation for {trafo_type} "{type_name}" could not be created. Error: {e}') 
                return 'FAILURE', None
            
    elif create_type == 'MERGE':
        try:
            entity = BusinessLandscapePycelonis.post_api_v2_workspaces_workspace_id_factories_sql(celonis.client, workspace_id, request_body = request_body)    # put back later
        except Exception as e:
            log_message(f'ERROR: Transformation for {trafo_type} "{type_name}" could not be created. Error: {e}') 
            return 'FAILURE', None
            
    
    #get trafos and update SQL(somehow was not able to add SQL in trafo creation call)
    sql_factory = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = entity.factory_id)
    try:
        factory_json = sql_factory.json_dict() 
    except:
        factory_json = {
            'factoryId': entity.factory_id,
            'description': '',
            'namespace': 'custom',
            'target': {'kind': trafo_type,
                       'entityRef': {'name': object_name, 'namespace': 'custom'}},
            'dataConnectionId': data_source_id,
            'displayName': display_name,
            'extensionSqlFactories': [],
            'localParameters': []}
    
    try:
        factory_json['transformations'][0]['property_sql_factory_datasets'] = [{ 
            'disabled': False, 
            'id': type_name,
            'namespace': 'custom',
            'completeOverwrite': False,
            'materialiseCte': False,
            'sql': sql_statement}]
    except:
        factory_json['transformations'] = [{
            'namespace': 'custom',
            'property_sql_factory_datasets': [{ 
            'disabled': False, 
            'id': type_name,
            'namespace': 'custom',
            'completeOverwrite': False,
            'materialiseCte': False,
            'sql': sql_statement}]
        }]
        
        
    factory_json = fixJSON(factory_json)
    try:
        del factory_json['changeDate'] 
    except:
        pass
    
    try:
        del factory_json['creationDate']
    except:
        pass
    #print(factory_json)
    
    try:
        entity = BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = entity.factory_id, request_body = factory_json)
        log_message(f'INFO: Transformation statement for {trafo_type} "{type_name}" successfully created.') 
        return 'SUCCESS', entity
    except Exception as e:
        log_message(f'ERROR: Transformation statement for {trafo_type} "{type_name}" could not be created. Error: {e}')  
        return 'FAILURE', None
    
def updateTransformationRelationship(factories, created_transformations, update_object, update_type, owner_object, key_object, owner_table, key_table):
    original_name = update_object['originalName']
    new_name = update_object['newName']
    log_message(f'INFO: Attempting to {update_type[0:-1]} relationship in transformation(s) for Object "{owner_object}" using table "{owner_table}" to Object "{key_object}" using table "{key_table}"...')
    try:
        try:
            factory = [factory for factory in factories if factory.target.entity_ref.name.lower() == owner_object.lower() and factory.factory_id in created_transformations][0]
            final_factory_list = [factory]
            if update_type == 'ignored':
                ignore_list = [original_name]
        except:
            update = False
            factory_list = [factory for factory in factories if factory.target.entity_ref.name.lower() == owner_object.lower()]
            if update_type == 'renamed':
                input_response = input(f'Object "{owner_object}" using table "{owner_table}" was skipped, would you like to {update_type[:-1]} use of object "{original_name}" to "{new_name}"?')
                if input_response.lower() in ('y','yes'):
                    update = True
                    
            elif update_type == 'ignored':
                if new_name == original_name:
                    input_response = input(f'Object "{owner_object}" using table "{owner_table}" was skipped, would you like to remove use of ignored object "{original_name}"?')
                    if input_response.lower() in ('y','yes'):
                        update = True
                        ignore_list = [original_name]
                else:
                    correct_response = False
                    input_message = f'Object "{owner_object}" using table "{owner_table}" was skipped, ignored object "{original_name}" was renamed to "{new_name}".'
                    input_message += f'\n\n\tType -1 to skip removal of ignored object.\n\tType 0 to remove use of both "{original_name}" and "{new_name}"\n\tType 1 to remove use of only original object name "{original_name}".\n\tType 2 to remove use of only new object name "{new_name}".\n\n'
                    input_response = input(input_message) #update to message that ignored object was renamed. -1 for none, 0 to remove use of both, 1 for original, 2 for rename
                    
                    while correct_response == False:
                        try:
                            try:
                                input_response = int(input_response.strip())
                            except:
                                raise Exception('')
                                
                            if input_reponse < -1 or input_response > 2:
                                raise Exception('')
                                
                            elif input_reponse == -1:
                                correct_response = True
                                update = False
                                
                            elif input_reponse == 0:
                                correct_response = True
                                update = True
                                ignore_list = [original_name, new_name]
                                
                            elif input_reponse == 1:
                                correct_response = True
                                update = True
                                ignore_list = [original_name]
                                
                            elif input_reponse == 2:
                                correct_response = True
                                update = True
                                ignore_list = [new_name]
                            
                        except Exception as e:
                            new_input = f'\nResponse "{input_response}" is not recognized. Please configure response based on the options listed above. {e}'
                            correct_response = False
                            input_responses = input(new_input)
                    
                    
            if update == True:
                if len(factory_list) == 1:
                    final_factory_list = factory_list

                if len(factory_list) > 1:
                    # add additional error handling
                    final_factory_list = []
                    correct_response = False
                    input_message = ''
                    i = 0
                    for factory in factory_list:
                        i += 1
                        data_source = [ds for ds in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_execution_factories_data_sources(celonis.client, workspace_id) if ds.data_source_id == factory.data_connection_id][0]
                        input_message += f'\n\t{i}. Transformation "{factory.display_name}" using data source "{data_source.display_name}"'
                    input_message += f'\n\nWARNING: Object "{owner_object}" using table "{owner_table}" was skipped and has multiple transformations\n\nType -1 to skip all.\nType 0 to update all.\nType the numbers of the transformations in the list above, separated by commas, to replace a subset.\n\n'
                    input_responses = input(input_message)

                    correct_response = False

                    while correct_response == False:
                        update_all = False
                        final_factory_list = []
                        input_responses_list = input_responses.strip().split(',')
                        try:
                            for input_response in input_responses_list:
                                try:
                                    input_response = int(input_response)
                                except Exception as e:
                                    raise Exception(f'Must be a list of integers.\n\n')

                                if input_response > len(factory_list):
                                    raise Exception(f'"{input_response}" does not correspond to field in list.\n\n')

                                elif input_response < 0:
                                    raise Exception(f'"{input_response}" is a negative integer.\n\n')

                                elif len(input_responses_list) > 1 and input_response == 0:
                                    raise Exception(f'"{input_response}" cannot be used with other values.\n\n')

                                elif len(input_responses_list) == 1 and input_response == 0:
                                    correct_response = True
                                    update_all = True
                                    final_factory_list = factory_list

                                elif input_response > 0 and pk_response <= len(factory_list):
                                    final_factory_list.append(factory_list[input_response-1])
                                    
                            if update_all == False:
                                correct_response = True

                        except Exception as e:
                            new_input = f'\nResponse "{str(input_responses)}" is not recognized. Please configure response based on the options listed above. {e}'
                            correct_response = False
                            input_responses = input(new_input)

                if len(factory_list) == 0:
                    raise Exception(f'ERROR: Object "{owner_object}" using table "{owner_table}" has no transformations.')
            else:
                final_factory_list = []
                
        for factory in final_factory_list:
            data_source = [ds for ds in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_execution_factories_data_sources(celonis.client, workspace_id) if ds.data_source_id == factory.data_connection_id][0]
            factory_json = json.loads(celonis.client.request(url = f'{celonis.client.base_url}/bl/api/v2/workspaces/{workspace_id}/factories/sql/{factory.factory_id}?environment=develop', method = 'GET').content)
            #sql_factory = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = factory.factory_id)
            #factory_json = sql_factory.json_dict() 
            sql_transformation = factory_json['transformations'][0]['propertySqlFactoryDatasets'][0]['sql']
            ## need to update the sql -- use different target object also depends on if removed or renamed -- DK NOTE 2024/11/18 -- done
            if update_type == 'ignored':
                for ignore_object in ignore_list:
                    sql_transformation = re.sub(f'\n\t,.+? AS "{ignore_object}"', '', sql_transformation)
            elif update_type == 'renamed':
                sql_transformation = sql_transformation.replace(f'AS "{original_name}"',f'AS "{new_name}"')


            factory_json['transformations'][0]['propertySqlFactoryDatasets'][0]['sql'] = sql_transformation

            #factory_json = fixJSON(factory_json)

            try:
                del factory_json['changeDate']
            except:
                pass
            try:
                del factory_json['changedBy']
            except:
                pass
            try:
                del factory_json['createdBy']
            except:
                pass
            try:
                del factory_json['creationDate']
            except:
                pass
            #del factory_json['factoryId']
            try:
                del factory_json['hasUserTemplate']
            except:
                pass
            try:
                del factory_json['disabled']
            except:
                pass
            try:
                del factory_json['draft']
            except:
                pass

            for t_i in range(len(factory_json['transformations'])):
                for p_i in range(len(factory_json['transformations'][t_i]['propertySqlFactoryDatasets'])):
                    try:
                        del factory_json['transformations'][t_i]['propertySqlFactoryDatasets'][p_i]['overwrite']
                    except:
                        pass
                    try:
                        del factory_json['transformations'][t_i]['propertySqlFactoryDatasets'][p_i]['type']
                    except:
                        pass

                for p_i in range(len(factory_json['transformations'][t_i]['relationshipTransformations'])):
                    for f_i in range(len(factory_json['transformations'][t_i]['relationshipTransformations'][p_i]['sqlFactoryDatasets'])):
                        try:
                            del factory_json['transformations'][t_i]['relationshipTransformations'][p_i]['sqlFactoryDatasets'][f_i]['overwrite']
                        except:
                            pass
                        try:
                            del factory_json['transformations'][t_i]['relationshipTransformations'][p_i]['sqlFactoryDatasets'][f_i]['type']
                        except:
                            pass
            #print(factory_json)
            try:
                BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = factory.factory_id, request_body = factory_json) 
                log_message(f'INFO: Relationship in transformation "{factory.display_name}" using data source "{data_source.display_name}" for Object "{owner_object}" using table "{owner_table}" to Object "{key_object}" using table "{key_table}" successfully {update_type}.')
            except Exception as e:
                log_message(f'ERROR: Relationship in transformation "{factory.display_name}" using data source "{data_source.display_name}" for Object "{owner_object}" using table "{owner_table}" to Object "{key_object}" using table "{key_table}" could not be {update_type}. Error: {e}')
            
    except Exception as e:
        log_message(f'ERROR: Relationship in transformation(s) for Object "{owner_object}" using table "{owner_table}" to Object "{key_object}" using table "{key_table}" could not be {update_type}. Error: {e}')
                        
        
def createO2Os(data_model, case_tables, activity_tables, final_relationship_sqls, update_objects_df, created_transformations): #need to evaluate this code!!!! update source and target object variables (affects ref object and everything else) and related transformations -- DK NOTE 2024/11/18
    perspective_objects = []
    factories = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_factories_sql(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content # update factories
    status = 'SUCCESS'
    for fk in data_model.foreign_keys: 
        try:
            if fk.target_table_id not in activity_tables and fk.source_table_id not in activity_tables: #if no join to activity table     
                target = data_model.get_table(fk.target_table_id) #n side
                
                target_source_table = target.name
                if target.alias != None and target.alias != '':
                    target_object = sanitizeName(target.alias, 'table') ## FIX
                else:
                    target_object = sanitizeName(target.name, 'table') ## FIX
                    
                try:
                    update_target_object = update_objects_df.query(f'originalName == "{target_object}"').iloc[0]
                    if update_target_object['originalName'] != update_target_object['newName']:
                        target_object = update_target_object['newName']

                    if update_target_object['updateType'] == 'REMOVE':
                        continue
                except:
                    pass
                
                
                source = data_model.get_table(fk.source_table_id) #1 side
                
                source_source_table = source.name
                if source.alias != None and source.alias != '':
                    source_object = sanitizeName(source.alias, 'table') ## FIX 
                else:
                    source_object = sanitizeName(source.name, 'table') ## FIX 
                    
                try:
                    update_source_object = update_objects_df.query(f'originalName == "{source_object}"').iloc[0]
                    if update_source_object['originalName'] != update_source_object['newName']:
                        source_object = update_source_object['newName']
                        update_type = 'renamed'
                        
                    if update_source_object['updateType'] == 'REMOVE':
                        update_type = 'ignored'
                        updateTransformationRelationship(factories, created_transformations, update_object = update_source_object, update_type = update_type, owner_object = target_object, key_object = source_object, owner_table = target_source_table, key_table = source_source_table)
                        continue
                    else:
                        updateTransformationRelationship(factories, created_transformations, update_object = update_source_object, update_type = update_type, owner_object = target_object, key_object = source_object, owner_table = target_source_table, key_table = source_source_table)
                except:
                    pass                    
                
                cardinality = 'HAS_ONE'
                owner = 'TARGET' ## need to update target object transformation with new source object -- DK NOTE 2024/11/18 -- DONE
                
                
            elif fk.target_table_id in activity_tables: #if 1:N join to activity table 
                target = data_model.get_table(fk.target_table_id) #n side - activity table
                source = data_model.get_table(fk.source_table_id) #1 side -- set source as owner
                
                
                source_source_table = source.name
                if source.alias != None and source.alias != '':
                    source_object = sanitizeName(source.alias, 'table') ## FIX 
                else:
                    source_object = sanitizeName(source.name, 'table') ## FIX 
                    
                try:
                    update_source_object = update_objects_df.query(f'originalName == "{source_object}"').iloc[0]
                    if update_source_object['originalName'] != update_source_object['newName']:
                        source_object = update_source_object['newName']

                    if update_source_object['updateType'] == 'REMOVE':
                        continue
                except:
                    pass  
                
                
                owner = 'SOURCE'
                
                if len([relationship for relationship in final_relationship_sqls if relationship['source_object'].lower() == source_object.lower() and relationship['activity_table'] == target]) > 0: #check if values in this table adhere to updates
                    relationship = [relationship for relationship in final_relationship_sqls if relationship['source_object'].lower() == source_object.lower() and relationship['activity_table'] == target][0]
                    target_object = relationship['target_object']
                    cardinality = 'HAS_MANY'
                    
                    try:
                        update_target_object = update_objects_df.query(f'originalName == "{target_object}"').iloc[0]
                        if update_target_object['originalName'] != update_target_object['newName']:
                            target_object = update_target_object['newName']
                            original_name = update_target_object['originalName']
                            relationship['sql'] = relationship['sql'].replace(f'AS "{original_name}"',f'AS "{target_object}"')

                        if update_target_object['updateType'] == 'REMOVE':
                            continue
                    except:
                        pass
                    
                    ###ADD SCRIPT TO UPDATE RELATIONSHIP FIELD ### NEED TO ACCOUNT FOR WHEN MULTIPLE TRANFORMS DUE TO MERGE -- NEED TO UPDATE THE RIGHT FACTORY -- DK NOTE 2024/11/18
                    factory = [factory for factory in factories if factory.target.entity_ref.name.lower() == source_object.lower() and factory.factory_id in created_transformations][0]
                    sql_factory = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = factory.factory_id)
                    factory_json = sql_factory.json_dict() 
                    relationship_transformations = factory_json['transformations'][0]['relationship_transformations']
                    
                    relationship_transformations.append(
                        {"relationshipName": target_object,
                        "sqlFactoryDatasets": [
                        { #'disabled': False, 
                        'id': target_object,
                        'namespace': 'custom',
                        #'completeOverwrite': False,
                        #'materialiseCte': False,
                        'sql': relationship['sql']}]}) ## need to update the sql -- use different target object also depends on if removed or renamed -- DK NOTE 2024/11/18 -- done
                    
                    factory_json['transformations'][0]['relationship_transformations'] = relationship_transformations
                    
                    factory_json = fixJSON(factory_json)
                    del factory_json['changeDate'] 
                    del factory_json['creationDate']
                    #print(factory_json)
                    try:
                        BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id = factory.factory_id, request_body = factory_json) 
                        log_message(f'INFO: Relationship Transformation for Object "{source_object}" using table "{source_source_table}" to Object "{target_object}" using table "{target_source_table}" relationship with cardinality "{cardinality}" successfully created.')
                    except Exception as e:
                        log_message(f'ERROR: Relationship Transformation for Object "{source_object}" using table "{source_source_table}" to Object "{target_object}" using table "{target_source_table}" relationship with cardinality "{cardinality}" could not be created. Error: {e}')
                        status = 'FAILURE'
                
                else:
                    cardinality = 'HAS_ONE'
                    owner = 'SOURCE'
                    
                    try:
                        case_table_id = [case_tables[case_table] for case_table in range(len(case_tables)) if activity_tables[case_table] == fk.target_table_id][0]
                        case_table = data_model.get_table(case_table_id)

                        if case_table.alias != None and case_table.alias != '':
                            case_table_object_name = case_table.alias
                        else:
                            case_table_object_name = case_table.name

                        target_object = sanitizeName(case_table_object_name, 'table')
                    except:
                        continue
                        
                    try:
                        update_target_object = update_objects_df.query(f'originalName == "{target_object}"').iloc[0]
                        if update_target_object['originalName'] != update_target_object['newName']:
                            target_object = update_target_object['newName']
                            update_type = 'renamed'

                        if update_target_object['updateType'] == 'REMOVE':
                            update_type = 'ignored'
                            updateTransformationRelationship(factories, created_transformations, update_object = update_target_object, update_type = update_type, owner_object = source_object, key_object = target_object, owner_table = source_source_table, key_table = target_source_table)
                            continue
                        else:
                            updateTransformationRelationship(factories, created_transformations, update_object = update_target_object, update_type = update_type, owner_object = source_object, key_object = target_object, owner_table = source_source_table, key_table = target_source_table)
                            
                    except:
                        pass
                    
                    ## need to update source transformation with new target object -- DK NOTE 2024/11/18 -- DONE
                    
                
            elif fk.source_table_id in activity_tables: #if N:1 join to activity table    
                target = data_model.get_table(fk.target_table_id) #n side
                
                target_source_table = target.name
                if target.alias != None and target.alias != '':
                    target_object = sanitizeName(target.alias, 'table') ## FIX
                else:
                    target_object = sanitizeName(target.name, 'table') ## FIX
                
                
                source = data_model.get_table(fk.source_table_id) #1 side -- activity table 
                source_source_table = source.name
                if source.alias != None and source.alias != '':
                    source_object = sanitizeName(source.alias, 'table') ## FIX 
                else:
                    source_object = sanitizeName(source.name, 'table') ## FIX 
                    
                
                try:
                    case_table_id = [case_tables[case_table] for case_table in range(len(case_tables)) if activity_tables[case_table] == fk.source_table_id][0]
                    case_table = data_model.get_table(case_table_id)


                    if case_table.alias != None and case_table.alias != '':
                        case_table_object_name = case_table.alias
                    else:
                        case_table_object_name = case_table.name
                    source_object = sanitizeName(case_table_object_name, 'table')
                    
                    try:
                        update_source_object = update_objects_df.query(f'originalName == "{source_object}"').iloc[0]
                        if update_source_object['originalName'] != update_source_object['newName']:
                            source_object = update_source_object['newName']

                        if update_source_object['updateType'] == 'REMOVE':
                            continue
                    except:
                        pass                    
                
                    cardinality = 'HAS_ONE'
                    owner = 'TARGET' ## need to update target object transformation with new source object -- DK NOTE 2024/11/18
                    
                    try:
                        update_target_object = update_objects_df.query(f'originalName == "{target_object}"').iloc[0]
                        if update_target_object['originalName'] != update_target_object['newName']:
                            target_object = update_target_object['newName']
                            update_type = 'renamed'

                        if update_target_object['updateType'] == 'REMOVE':
                            update_type = 'ignored'
                            updateTransformationRelationship(factories, created_transformations, update_object = update_target_object, update_type = update_type, owner_object = source_object, key_object = target_object, owner_table = source_source_table, key_table = target_source_table)
                            continue
                        else:
                            updateTransformationRelationship(factories, created_transformations, update_object = update_target_object, update_type = update_type, owner_object = source_object, key_object = target_object, owner_table = source_source_table, key_table = target_source_table)
                    except:
                        pass
                    
                except:
                    continue
                
            if source_object.lower() != target_object.lower():

                objects = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content

                ### NEED TO UPDATE REF OBJECT IF RENAME OR REMOVED (and transformation) -- DK NOTE 2024/11/18 -- DONE
                
                if owner == 'TARGET':
                    object_data = [o for o in objects if target_object.lower() == o.name.lower()][0]
                    ref_object = source_object
                if owner == 'SOURCE':
                    object_data = [o for o in objects if source_object.lower() == o.name.lower()][0]
                    ref_object = target_object

                object_id = object_data.id
                request_body = object_data.json_dict()
                object_relationships = request_body['relationships']
                object_relationship = {
                    'name': ref_object,
                    'namespace': 'custom',
                    'cardinality': cardinality,
                    'target': {
                        'object_ref' : {
                            'name' : ref_object,
                            'namespace' : 'custom'
                        }
                    }  
                }
                
                if ref_object.lower() not in [object_relationship["target"]["object_ref"]["name"].lower() for object_relationship in object_relationships]:
                    object_relationships.append(object_relationship)
                
                    request_body['relationships'] = object_relationships
                    request_body = fixJSON(request_body)
                    del request_body['changeDate']
                    del request_body['creationDate']
                    del request_body['changedBy']
                    del request_body['createdBy']
                    del request_body['id']
                    del request_body['description']
                    BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_type_id = object_id, request_body = request_body)
                    log_message(f'INFO: Object "{source_object}" using table "{source_source_table}" to Object "{target_object}" using table "{target_source_table}" relationship with cardinality "{cardinality}" successfully created.')
                
                if object_data.name.lower() in [perspective_object['name'].lower() for perspective_object in perspective_objects]:
                    range_loc = [perspective_object for perspective_object in range(len(perspective_objects)) if perspective_objects[perspective_object]['name'].lower() == object_data.name.lower()][0]
                    perspective_object = perspective_objects[range_loc]
                    perspective_object_relationships = perspective_object['relationships']
                    perspective_object_relationships.append({
                          'name': ref_object,
                          'namespace': 'custom',
                          'strategy': 'LINK'
                    })
                    perspective_object['relationships'] = perspective_object_relationships
                    perspective_objects[range_loc] = perspective_object
                else:
                    perspective_objects.append({
                        'name': object_data.name,
                        'namespace': 'custom',
                        'relationships': [{
                          'name': ref_object,
                          'namespace': 'custom',
                          'strategy': 'LINK'
                    }]
                        })

        except Exception as e:
            log_message(f'ERROR: Object "{source_object}" using table "{source_source_table}" to Object "{target_object}" using table "{target_source_table}" relationship with cardinality "{cardinality}" could not be created. Error: {e}')
            status = 'FAILURE'
            
    return perspective_objects, status
                
def createEvent(event_name, fields, data_model, case_tables):
    ### UPDATE TO LOOP THROUGH "CASE" TABLES
    relationships = []
    for case_table in case_tables: 
        relationships.append({
            'cardinality': 'HAS_ONE',
            'name': case_table['ocName'], #potential for relationship field name to match existing field -- add logic to handle
            'namespace': 'custom',
            'target': {
                'mappedBy': '',
                'mappedByNamespace': '',
                'objectRef': {
                    'name': case_table['ocName'],
                    'namespace': 'custom'
                }
            }
        })
    
    request_body = {
        'name': event_name,
        'tags': [],
        'categories': [{'metadata': {'name': 'Processes', 'namespace': 'celonis'},
          'values': [{'displayName': 'Compatibility Mode',
            'name': 'CompatibilityMode',
            'namespace': 'custom'}]}],
        'relationships': relationships,
        'fields': fields,
        'color': '#0A18BB'}
        
    events = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_events(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content
    if event_name in [e.name for e in events]:
        log_message(f'WARNING: Event "{event_name}" already exists. Attempting to update...')
        event_id = [e.id for e in events if event_name == e.name][0]
                
        try:
            entity = BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_types_events_event_type_id(celonis.client, workspace_id, event_type_id = event_id, request_body = request_body)
            log_message(f'INFO: Event "{event_name}" successfully updated.')
            return 'SUCCESS', entity
        except Exception as e:
            log_message(f'ERROR: Event "{event_name}" could not be updated. Error: {e}')
            return 'FAILURE', None
            
    else:
        try:
            entity = BusinessLandscapePycelonis.post_api_v2_workspaces_workspace_id_types_events(celonis.client, workspace_id, request_body = request_body) 
            log_message(f'INFO: Event "{event_name}" successfully created.')
            return 'SUCCESS', entity
        except Exception as e:
            log_message(f'ERROR: Event "{event_name}" could not be created. Error: {e}')
            return 'FAILURE', None
        
def createPerspective(data_model, perspective_objects, activity_tables, update_objects_df):     
    perspective_name = sanitizeName(data_model.name, 'table')
    
    for table in data_model.get_tables(): #add remaining tables with no relationships
        if table.alias != None and table.alias != '':
            object_name = sanitizeName(table.alias, 'table')
        else:
            object_name = sanitizeName(table.name, 'table')
            
        try:
            update_object = update_objects_df.query(f'originalName == "{object_name}"').iloc[0]
            if update_object['originalName'] != update_object['newName']:
                object_name = update_object['newName']

            if update_object['updateType'] == 'REMOVE':
                continue
        except:
            pass
            
        if object_name not in [table["name"] for table in perspective_objects] and table.id not in activity_tables:
            
            perspective_objects.append({
                "namespace": "custom",
                "name": object_name,
                "relationships": []
            })
    
    request_body = {
            'name': perspective_name,
            'objects': perspective_objects,
            'projections': [],
            'tags': []
        }

    request_body = fixJSON(request_body)
    
    perspectives = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_perspectives(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content
    #data_model.name
    if perspective_name in [perspective.name for perspective in perspectives]:
        log_message(f'WARNING: Perspective "{perspective_name}" already exists. Attempting to update...')
        perspective_id = [perspective.id for perspective in perspectives if perspective.name == perspective_name][0]
        try:
            entity = BusinessLandscapePycelonis.put_api_v2_workspaces_workspace_id_perspectives_perspective_id(celonis.client, workspace_id, perspective_id = perspective_id, request_body = request_body)
            log_message(f'INFO: Perspective "{perspective_name}" successfully updated.')
            return 'SUCCESS', entity
        except Exception as e:
            log_message(f'ERROR: Perspective "{perspective_name}" could not be updated. Error: {e}')
            return 'FAILURE', None
            
    else:
        try:
            entity = BusinessLandscapePycelonis.post_api_v2_workspaces_workspace_id_perspectives(celonis.client, workspace_id, request_body = request_body)
            log_message(f'INFO: Perspective "{perspective_name}" successfully created.')
            return 'SUCCESS', entity
        except Exception as e:
            log_message(f'ERROR: Perspective "{perspective_name}" could not be created. Error: {e}')
            return 'FAILURE', None
        
def deleteObjectsandEvents(created):
    for perspective_id in created['perspectives']:
        try:
            perspective_name = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_perspectives_perspective_id(celonis.client, workspace_id, perspective_id).name
            try:
                BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_perspectives_perspective_id(celonis.client, workspace_id, perspective_id)
                log_message(f'INFO: Perspective "{perspective_name}" with ID "{perspective_id}" successfully deleted.')
            except Exception as e:
                log_message(f'ERROR: Perspective "{perspective_name}" with ID "{perspective_id}" could not be deleted. Error: {e}')
        except:
            continue
    
    
    for factory_id in created['transformations']:
        try:
            BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_factories_sql_factory_id(celonis.client, workspace_id, factory_id)
            log_message(f'INFO: Transformation "{factory_id}" successfully deleted.')
        except Exception as e:
            log_message(f'ERROR: Transformation "{factory_id}" could not be deleted. Error: {e}')
            
    for event_id in created['events']:
        try:
            event_name = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_events_event_type_id(celonis.client, workspace_id, event_id).name
            try:
                BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_types_events_event_type_id(celonis.client, workspace_id, event_id)
                log_message(f'INFO: Event "{event_name}" with ID "{event_id}" successfully deleted.')
            except Exception as e:
                log_message(f'ERROR: Event "{event_name}" with ID "{event_id}" could not be deleted. Error: {e}')
        except:
            continue

    for object_id in created['objects']:
        try:
            object_name = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_id).name
        except:
            continue

        try:
            object_relationships = BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects_object_type_id_relationships(celonis.client, workspace_id, object_id).content
            for relationship in object_relationships:
                if relationship.target.object.name == object_name:
                    try:
                        if relationship.owner == 'SOURCE':
                            owner_object_id = [source.id for source in BusinessLandscapePycelonis.get_api_v2_workspaces_workspace_id_types_objects(celonis.client, workspace_id,  pagination={"requestMode": "ALL"}).content if source.name == relationship.source.object.name][0]
                            relationship_name = relationship.source.relationship.name
                        if relationship.owner == 'TARGET':
                            owner_object_id = object_id
                            relationship_name = relationship.target.relationship.name
                        
                        BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_types_objects_object_type_id_relationships_target_type_id(celonis.client, workspace_id, object_type_id = owner_object_id, target_type_id = object_id, relationship = relationship_name)
                        log_message(f'INFO: Object relationship "{relationship.source.object.name}" to "{relationship.target.object.name}" with cardinality "{relationship.cardinality}" successfully deleted.')
                    except Exception as e:
                        log_message(f'ERROR: Object relationship "{relationship.source.object.name}" to "{relationship.target.object.name}" with cardinality "{relationship.cardinality}" could not be deleted. Error: {e}')
        except:
            pass

        try:
            BusinessLandscapePycelonis.delete_api_v2_workspaces_workspace_id_types_objects_object_type_id(celonis.client, workspace_id, object_id)
            log_message(f'INFO: Object "{object_name}" with ID "{object_id}" successfully deleted.')
        except Exception as e:
            log_message(f'ERROR: Object "{object_name}" with ID "{object_id}" could not be deleted. Error: {e}')
    

def dm_to_ocdm(celonis, data_model, delete_if_error = False):
    created = {'objects': [],
              'events': [],
              'transformations': [],
              'perspectives':[]}
    data_pool_id = data_model.data_pool_id 
    data_pool = celonis.data_integration.get_data_pool(data_pool_id) 
    data_pool_name = data_pool.name 
    case_tables = [process_config.case_table_id for process_config in data_model.process_configurations]
    activity_tables = [process_config.activity_table_id for process_config in data_model.process_configurations] #fixed to work for multiple eventlogs in same DM
    objects = []
    events = []
    final_relationship_sqls = []
    object_primary_keys = []
    
    ### 2024/10/25 DK UPDATES START
    
    update_objects_df = pd.DataFrame(columns = ['ccTableName', 'ccTableAlias', 'originalName', 'newName', 'updateType'])
    sql_mapping_df = pd.DataFrame(columns = ['cc_table', 'cc_alias','cc_column','oc_table','oc_column'])
    
    objects_df = pd.DataFrame(columns = ['tableName', 'tableAlias', 'tableId','objectNameRaw','nameSanitized', 'nameSanitizedIgnoreCase', 'ignore'])
    for table in [table for table in data_model.get_tables() if table.id not in activity_tables]:
        if table.alias != None and table.alias != '':
            objects_df.loc[len(objects_df)] = [table.name, table.alias, table.id, table.alias, sanitizeName(table.alias, 'table'), sanitizeName(table.alias, 'table').lower(), False]
        else:
            objects_df.loc[len(objects_df)] = [table.name, table.alias, table.id, table.name, sanitizeName(table.name, 'table'), sanitizeName(table.name, 'table').lower(), False]
            
    objects_df = removeDupes(df = objects_df, name_type = 'table')

    ### 2024/10/25 DK UPDATES END
    
    for table in data_model.get_tables(): #have to do event tables first otherwise could cause problems when setting event
        table_id = table.id
        table_name = table.name
        table_alias = table.alias  ## DK ADDED 5/6/2024 --
        cc_data_source_id = table.data_source_id
        if cc_data_source_id == None:
            schema_name = ''
            schema_param = ''
        else:
            schema_name = data_pool.get_data_connection(cc_data_source_id).name
            schema_param = f'<%=DATASOURCE:{schema_name.upper()}%>'
        
        # if table_alias != None and table_alias != '': ## DK ADDED 5/6/2024 --
        #     object_name_raw = table_alias 
        # else:
        #     object_name_raw = table_name
        
        isActivityTable = table.id in activity_tables
        
        if not isActivityTable:
            
            data_source_id, available_columns, final_source_table = checkTableExistence(data_pool, table)
            
            type_name = objects_df.query(f'tableId == "{table_id}"')['nameSanitized'].iloc[0]
            ignore_table = objects_df.query(f'tableId == "{table_id}"')['ignore'].iloc[0]
            #print(table_name, type_name)
            
            ## FIND TRANFORMATION FOR TABLE 
            
            ## BUILD DATAFRAME FOR OBJECT SOURCE TABLES FOR EVENTS -- table in data model -- any table where the ID is coming from in the source table 
            if ignore_table == False:
                
                fields = create_fields(data_model, table_id, available_columns)
                #print(type_name, sql_statement)

                objects.append(type_name) 
                status, entity, final_object_name, fields = createObject(type_name, fields, objects_df)
                
                if status == 'FAILURE' and delete_if_error == True:
                    deleteObjectsandEvents(created)
                elif status in ('MERGE', 'REPLACE', 'SUCCESS'):
                    created['objects'].append(entity.id)
                    
                if type_name != final_object_name or status == 'IGNORE':
                    if status == 'IGNORE':
                        update_type = 'REMOVE'
                    else:
                        update_type = 'RENAME'
                    update_objects_df.loc[len(update_objects_df)] = [table_name, table_alias, type_name, final_object_name, update_type]
                
                if status not in ('SKIP', 'IGNORE'):
                    sql_mapping = createSQLMapping(entity_name = final_object_name, fields = fields) #should we include sql mapping for skipped objects?
                    sql_mapping['cc_alias'] = table_alias
                    sql_mapping_df = pd.concat([sql_mapping_df, sql_mapping])
                    
                    sql_statement, fields, relationship_table_sqls, primary_key_columns = assembleSQL(data_model, table_id, final_source_table.name, final_object_name, available_columns, isActivityTable, case_tables, activity_tables, fields) ## DK UPDATED 5/6/2024
                    final_relationship_sqls += relationship_table_sqls
                    
                    status, entity = createTrafo(final_object_name, data_source_id, sql_statement, isActivityTable, status)  
                    
                    if status == 'FAILURE' and delete_if_error == True:
                        deleteObjectsandEvents(created)
                    elif status == 'SUCCESS':
                        created['transformations'].append(entity.factory_id)
                elif status == 'SKIP':
                    ## NEED TO UPDATE SQL TO REMOVE SKIPPED PKS from formula
                    log_message(f'INFO: Object "{final_object_name}" skipped.')
                    
                elif status == 'IGNORE':
                    ## NEED TO UPDATE SQL TO REMOVE SKIPPED PKS from formula
                    log_message(f'INFO: Object "{final_object_name}" ignored.')
                
    for table in data_model.get_tables(): 
        table_id = table.id
        table_name = table.name
        isActivityTable = table.id in activity_tables
        
        table_alias = table.alias  ## DK ADDED 5/6/2024 --
        
        if table_alias != None and table_alias != '': ## DK ADDED 5/6/2024 --
            event_name_raw = table_alias 
        else:
            event_name_raw = table_name
        
        if isActivityTable:
            data_source_id, available_columns, final_source_table = checkTableExistence(data_pool, table)
            type_name = sanitizeName(event_name_raw, 'table') ## DK UPDATED 5/6/2024 -- 
            
            ## FIND TRANFORMATIONS FOR TABLE 
            
            #case_table = [table.name for table in data_model.get_tables() if table.id == [case_tables[case_table] for case_table in range(len(case_tables)) if activity_tables[case_table] == table.id][0]]
            try:
                case_table_id = [case_tables[case_table] for case_table in range(len(case_tables)) if activity_tables[case_table] == table.id][0]
                case_table = data_model.get_table(case_table_id)  ## NEED TO UPDATE
            except:
                case_table = None
                
            case_table_objects = []
            
            if case_table != None:
                if case_table.alias != None and case_table.alias != '':
                    case_table_name = case_table.alias
                else:
                    case_table_name = case_table.name
                    
                case_table_object_name = sanitizeName(case_table_name, 'table')
                
                try:
                    update_object = update_objects_df.query(f'originalName == "{case_table_object_name}"').iloc[0]
                    if update_object['originalName'] != update_object['newName']:
                        case_table_object_name = update_object['newName']
                    
                    if update_object['updateType'] == 'REMOVE':
                        log_message(f'WARNING: Event "{type_name}" has relationship to removed object "{case_table_object_name}". Ignoring relationship...')
                    else:
                        case_table_objects.append({'tableId': case_table.id, 'sqlName': case_table.name, 'ccName': case_table_name, 'ocName': case_table_object_name})
                except:
                    case_table_objects.append({'tableId': case_table.id, 'sqlName': case_table.name, 'ccName': case_table_name, 'ocName': case_table_object_name})
            
                
            else:
                for fk in data_model.foreign_keys:
                    if fk.target_table_id == table_id:

                        case_table_id = fk.source_table_id
                        case_table = data_model.get_table(case_table_id)

                        if case_table.alias != None and case_table.alias != '':
                            case_table_name = case_table.alias
                        else:
                            case_table_name = case_table.name

                        case_table_object_name = sanitizeName(case_table_name, 'table')

                        try:
                            update_object = update_objects_df.query(f'originalName == "{case_table_object_name}"').iloc[0]
                            if update_object['originalName'] != update_object['newName']:
                                case_table_object_name = update_object['newName']

                            if update_object['updateType'] == 'REMOVE':
                                log_message(f'WARNING: Event "{type_name}" has relationship to removed object "{case_table_object_name}". Ignoring relationship...')
                            else:
                                case_table_objects.append({'tableId': case_table.id, 'sqlName': case_table.name, 'ccName': case_table_name, 'ocName': case_table_object_name})
                        except:
                            case_table_objects.append({'tableId': case_table.id, 'sqlName': case_table.name, 'ccName': case_table_name, 'ocName': case_table_object_name})
                                            
            if len(case_table_objects) == 0:
                log_message(f'WARNING: Event "{type_name}" has no relationship to any objects. Skipping event creation...')
                continue
                
            else:
                fields = create_fields(data_model, table_id, available_columns)
                #print(table_name, type_name)
                #print(type_name, sql_statement)
                sql_mapping = createSQLMapping(entity_name = type_name, fields = fields)
                sql_mapping['cc_alias'] = table_alias
                sql_mapping_df = pd.concat([sql_mapping_df, sql_mapping])


                status, entity = createEvent(type_name, fields, data_model, case_table_objects)
                if status == 'FAILURE' and delete_if_error == True:
                    deleteObjectsandEvents(created)
                elif status == 'SUCCESS':
                    created['events'].append(entity.id)

                    sql_statement, fields, relationship_table_sqls, primary_key_columns = assembleSQL(data_model, table_id, final_source_table.name, type_name, available_columns, isActivityTable, case_table_objects, activity_tables, fields) ## FIX 
                    status, entity = createTrafo(type_name, data_source_id, sql_statement, isActivityTable, 'REPLACE')  #question becomes -- should a table also tied to the activity table have a relationship to the events?
                    if status == 'FAILURE' and delete_if_error == True:
                        deleteObjectsandEvents(created)
                    elif status == 'SUCCESS':
                        created['transformations'].append(entity.factory_id)
        # except Exception as e:
        #     print(f'Table {table_name} could not be migrated. Error:', e)
        #     break

    perspective_objects, status = createO2Os(data_model, case_tables, activity_tables, final_relationship_sqls, update_objects_df, created['transformations']) 
    if status == 'FAILURE' and delete_if_error == True:
        deleteObjectsandEvents(created)
    
    status, entity = createPerspective(data_model, perspective_objects, activity_tables, update_objects_df) 
    if status == 'FAILURE' and delete_if_error == True:
        deleteObjectsandEvents(created)
    elif status == 'SUCCESS':
        created['perspectives'].append(entity.id)        
    
    return created, sql_mapping_df


def createPQLPreamble(data_model, file_name='PQL_Preamble.txt'):
    activity_tables = [process_config.activity_table_id for process_config in data_model.process_configurations]
    case_tables = [process_config.case_table_id for process_config in data_model.process_configurations if process_config.case_table_id != None]
    
    #Create text file to write to
    with open(file_name, "w") as file:
        for table in data_model.get_tables():
            table_name = table.name
            if table.alias is not None and table.alias != '':
                table_name = table.alias
            else:
                table_name = table.name
            
            table_name_OCPM = sanitizeName(table_name,'table')

            isActivityTable = table.id in activity_tables
            if not isActivityTable:
                file.write(f'REGISTER "{table_name}" AS "o_custom_{table_name_OCPM}";\n')
            else:
                activity_table_id = table.id
                
                found_fk = False
                for case_table in case_tables:
                    fk = [sfk for sfk in data_model.foreign_keys if sfk.source_table_id == case_table and sfk.target_table_id == activity_table_id]
                    if len(fk) > 0:
                        found_fk = True
                        case_table = data_model.get_table(fk[0].source_table_id)
                        
                        if case_table.alias != None and case_table.alias != '':
                            case_table_name = case_table.alias
                        else:
                            case_table_name = case_table.name
                         
                        activity_table_name = 'el_custom_' + table_name_OCPM + '_' + sanitizeName(case_table_name,'table')
                        case_table_name_OCPM = 'o_custom_' + sanitizeName(case_table_name, 'table')
                        event_name_OCPM = 'e_custom_' + table_name_OCPM
                        break
                if found_fk == False:
                    activity_table_name = '<ACTIVITY_TABLE>'
                    log_message(f'INFO: Did not find Case Table related to Activity table "{table_name}", please replace <ACTIVITY_TABLE> manually in file.')
                if len(activity_tables) >= 1 and found_fk == True:
                    file.write(f'REGISTER "{activity_table_name}" AS CREATE_EVENTLOG("{case_table_name_OCPM}", INCLUDE ["{event_name_OCPM}"]);\n')
                else:
                    file.write(f'REGISTER "{table_name}" AS "{activity_table_name}";\n')
                file.write(f'REGISTER "{table_name}" AS "{activity_table_name}";\n')
            
            for column in table.get_columns():
                column_name = column.name
                column_name_OCPM = sanitizeName(column_name,'column')
                
                if column_name.lower() == 'eventtime':
                    file.write(f'EXTEND "{activity_table_name}" WITH "{column_name}" AS "{activity_table_name}"."Timestamp";\n')
                
                elif column_name.lower() != column_name_OCPM.lower():
                    if isActivityTable:
                        
                        file.write(f'EXTEND "{activity_table_name}" WITH "{column_name}" AS "{activity_table_name}"."{column_name_OCPM}";\n')
                    else:
                        file.write(f'EXTEND "o_custom_{table_name_OCPM}" WITH "{column_name}" AS "o_custom_{table_name_OCPM}"."{column_name_OCPM}";\n')



# Main Flow

### Setting Up Logger

In [ ]:
log = setup_logger()

### Getting Workspace and Setting Up Celonis Object

In [ ]:
celonis = get_celonis(url, apiKey)
pool = celonis.data_integration.get_data_pool(sourceDataPoolID)
data_model = pool.get_data_model(dataModelID)

target_pool = celonis.data_integration.get_data_pool(targetDataPoolID)

def get_ocpm_workspace(pool_id = None, data_model_id = None):
    if pool_id != None:
        try:
            pool = celonis.data_integration.get_data_pool(pool_id)
        except:
            raise Exception(f'Could not find data pool "{pool_id}". Please check ID.')
        try:
            workspace = [workspace for workspace in celonis.client.request(url = f'{celonis.client.base_url}/integration/api/pools/paged?limit=1000000&page=0&sort=name&ascending=true', method = 'GET').json()['content'] if workspace['name'] == pool.name][0]
        except:
            raise Exception(f'Could not find workspace using "{pool.name}" Data Pool. Please check if Objects and Events have been set up for this data pool.')
    elif data_model_id != None:
        try:
            pool = [pool for pool in celonis.data_integration.get_data_pools() if len([dm for dm in pool.get_data_models() if dm.id == dataModelID]) > 0][0]
        except:
            raise Exception(f'Could not find data model "{data_model_id}". Please check ID.')
        
        try:
            workspace = [workspace for workspace in celonis.client.request(url = f'{celonis.client.base_url}/integration/api/pools/paged?limit=1000000&page=0&sort=name&ascending=true', method = 'GET').json()['content'] if workspace['name'] == pool.name][0]
        except:
            raise Exception(f'Could not find workspace using "{pool.name}" Data Pool. Please check if Objects and Events have been set up for this data pool.')
    else:
        try:
            workspace = [workspace for workspace in celonis.client.request(url = f'{celonis.client.base_url}/integration/api/pools/paged?limit=1000000&page=0&sort=name&ascending=true', method = 'GET').json()['content'] if workspace['name'] == 'OCPM Data Pool'][0]
        except:
            raise Exception(f'Could not find workspace for "OCPM Data Pool".')
            
    return workspace
        
workspace = get_ocpm_workspace(pool_id = targetDataPoolID, data_model_id = dataModelID) 
workspace_id = workspace['id']
print(workspace['name'])
workspace_name = workspace['name']

### Migration

In [ ]:
# Run script
created, sql_mapping_df = dm_to_ocdm(celonis, data_model, delete_if_error = False)
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
sql_mapping_df.to_csv(f'{team_name}_{process_name}_{timestamp}_sql_mapping.csv')

# Genarate PQL Preamble
createPQLPreamble(data_model, file_name=f'PQL_Preamble_{team_name}_{process_name}_{timestamp}.txt')